# ViC-MAE: Temporal Extension Training on Kaggle GPU (Resume from Epoch 10)

**Architecture:** ViC-MAE ViT-Base with SiamMAE-Style Cross-Frame Reconstruction & Teacher Feature Preservation

### Features Included in this Run:
1. **Checkpoint Resume (Epochs 10 to 30):** Continues joint fine-tuning from `checkpoint_epoch0009_s2.pth` (epochs 0-9 completed).
2. **Continuous Time-Gap Embedding & Cross-Attention Decoder:** Queries target frame patches (heavily masked at ~95%) conditioned on context frame tokens + time gap $\Delta t$.
3. **Feature Preservation Loss ($L_{pres}$):** Freezes pretrained ViC-MAE teacher and penalizes cosine drift on active encoder tokens.
4. **Full Mixed-Precision (`--use_amp`) & LoRA ($r=16$) Support:** Optimized for Kaggle T4 / P100 GPUs.

### Kaggle Checkpoint Setup:
- Uploaded checkpoint dataset: `vicmae-checkpoint` (contains `checkpoint_epoch0009_s2.pth`).
- Path auto-detection supports both `/kaggle/input/datasets/tirthmanavadariya/vicmae-checkpoint/checkpoint_epoch0009_s2.pth` and `/kaggle/input/vicmae-checkpoint/checkpoint_epoch0009_s2.pth`.
- Accelerator: GPU T4 x2 (or P100), Internet: On.

## 1. Environment & GPU Setup

In [ ]:
!pip install -q timm==0.4.12 gdown pytest wandb matplotlib

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU VRAM:        {vram:.1f} GB")
else:
    print("WARNING: No GPU detected! Please enable GPU accelerator in Kaggle settings.")

## 2. Clone Base Repo, Pretrained Checkpoint & Tiny-ImageNet Dataset

In [ ]:
import os, urllib.request, zipfile, shutil
from pathlib import Path

WORK_DIR = "/kaggle/working"
REPO_DIR = os.path.join(WORK_DIR, "ViC-MAE")
CKPT_PATH = os.path.join(WORK_DIR, "vicmae_pretrain_vit_base_in1k_k400.pth")
DATA_DIR = os.path.join(WORK_DIR, "tiny-imagenet-200")
ZIP_PATH = os.path.join(WORK_DIR, "tiny-imagenet-200.zip")

# 1. Clone base repo
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/jeffhernandez1995/ViC-MAE {REPO_DIR}")
    print("✅ Base ViC-MAE repo cloned")
else:
    print("✅ Base ViC-MAE repo already exists")

# 2. Download Pretrained ViC-MAE ViT-B checkpoint (~327 MB) from official Google Drive
GDRIVE_ID = "1p879SxrProg_ttPKvVXqJNpUvuCLkjAf"
if not os.path.exists(CKPT_PATH) or os.path.getsize(CKPT_PATH) < 1000000:
    print("Downloading ViC-MAE ViT-B/16 checkpoint...")
    ret = os.system(f"gdown --id {GDRIVE_ID} -O {CKPT_PATH} --fuzzy")
    if ret != 0 or not os.path.exists(CKPT_PATH) or os.path.getsize(CKPT_PATH) < 1000000:
        direct_url = f"https://drive.usercontent.google.com/download?id={GDRIVE_ID}&export=download&confirm=t"
        os.system(f"curl -L -A 'Mozilla/5.0' '{direct_url}' -o {CKPT_PATH}")
    print(f"✅ Checkpoint downloaded: {os.path.getsize(CKPT_PATH) / 1e6:.1f} MB")
else:
    print("✅ Checkpoint already exists")

# 3. Download Tiny-ImageNet-200 (~248 MB)
TINY_URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
if not os.path.exists(DATA_DIR):
    if not os.path.exists(ZIP_PATH):
        print("Downloading Tiny-ImageNet-200...")
        urllib.request.urlretrieve(TINY_URL, ZIP_PATH)
    print("Extracting Tiny-ImageNet...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(WORK_DIR)
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
    print("✅ Tiny-ImageNet extracted")
else:
    print("✅ Tiny-ImageNet already exists")

# 4. Reorganize into standard torchvision ImageFolder structure
train_dir = Path(DATA_DIR) / "train"
for cls_dir in train_dir.iterdir():
    if not cls_dir.is_dir():
        continue
    images_subdir = cls_dir / "images"
    if images_subdir.exists():
        for img in images_subdir.iterdir():
            img.rename(cls_dir / img.name)
        images_subdir.rmdir()
    for txt in cls_dir.glob("*.txt"):
        txt.unlink()

val_dir = Path(DATA_DIR) / "val"
annotations_file = val_dir / "val_annotations.txt"
if annotations_file.exists():
    img_to_class = {}
    with open(annotations_file) as f:
        for line in f:
            parts = line.strip().split('\t')
            img_to_class[parts[0]] = parts[1]
    images_dir = val_dir / "images"
    for img_name, cls_id in img_to_class.items():
        cls_dir = val_dir / cls_id
        cls_dir.mkdir(exist_ok=True)
        src = images_dir / img_name
        if src.exists():
            shutil.move(str(src), str(cls_dir / img_name))
    if images_dir.exists():
        shutil.rmtree(images_dir)
    annotations_file.unlink()

print(f"✅ Dataset verified: {len(list(train_dir.iterdir()))} train classes ready.")

## 3. Deploy ViC-MAE Extension Modules
Inject the newly implemented extension modules into the cloned `ViC-MAE` directory.

In [ ]:
import os

REPO_DIR = "/kaggle/working/ViC-MAE"
os.makedirs(os.path.join(REPO_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(REPO_DIR, "datasets"), exist_ok=True)

files = {
    os.path.join(REPO_DIR, "models", "__init__.py"): '\ufeff"""Model package for the extended ViC-MAE codebase."""\nfrom models.cross_decoder import CrossAttentionBlock, CrossFrameDecoder\nfrom models.extended_vicmae import ExtendedViCMAE\nfrom models.lora import LoRALinear, inject_lora, lora_state_dict, mark_only_lora_trainable\n\n__all__ = [\n    "CrossAttentionBlock",\n    "CrossFrameDecoder",\n    "ExtendedViCMAE",\n    "LoRALinear",\n    "inject_lora",\n    "lora_state_dict",\n    "mark_only_lora_trainable",\n]\n',
    os.path.join(REPO_DIR, "models", "cross_decoder.py"): '"""\nmodels/cross_decoder.py\n-----------------------\nSiamMAE-style Cross-Frame Decoder for ViC-MAE temporal extension.\n\nArchitecture overview:\n  - CrossAttentionBlock  : single Transformer block where queries come from\n                           target (masked) patches and keys/values come from\n                           the context-frame encoder tokens.\n  - CrossFrameDecoder    : 4-block cross-attention decoder that reconstructs\n                           heavily masked Frame-j patches conditioned on\n                           Frame-i encoder tokens + a continuous Delta-t embedding.\n\nReferences:\n  - SiamMAE  : https://arxiv.org/abs/2212.05543\n  - MAE       : https://arxiv.org/abs/2111.06377\n  - ViC-MAE   : internal codebase (models_vicmae.py)\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef _get_2d_sincos_pos_embed(embed_dim: int, grid_size: int) -> torch.Tensor:\n    """Return (grid_size*grid_size, embed_dim) 2-D sin-cos positional embedding."""\n    assert embed_dim % 2 == 0, "embed_dim must be even."\n    grid_h = np.arange(grid_size, dtype=np.float32)\n    grid_w = np.arange(grid_size, dtype=np.float32)\n    grid_w, grid_h = np.meshgrid(grid_w, grid_h)          # (grid_size, grid_size) each\n    grid = np.stack([grid_h.ravel(), grid_w.ravel()], axis=0)  # (2, L)\n\n    # Use embed_dim // 4 frequencies per axis (4 embeddings total: sin/cos for h and w)\n    half = embed_dim // 2\n    omega = np.arange(half // 2, dtype=np.float32) / (half // 2)\n    omega = 1.0 / (10000 ** omega)                        # (half/2,)\n\n    pos_h = np.einsum("m,d->md", grid[0], omega)          # (L, half/2)\n    pos_w = np.einsum("m,d->md", grid[1], omega)          # (L, half/2)\n    emb = np.concatenate([np.sin(pos_h), np.cos(pos_h),\n                          np.sin(pos_w), np.cos(pos_w)], axis=1)  # (L, embed_dim)\n    return torch.from_numpy(emb).float()\n\n\n# ---------------------------------------------------------------------------\n# CrossAttentionBlock\n# ---------------------------------------------------------------------------\n\nclass CrossAttentionBlock(nn.Module):\n    """\n    Transformer block where:\n      - queries   = target (masked) patch tokens.\n      - keys/vals = context-frame encoder output tokens.\n\n    Architecture:\n      1. Cross-Attention  (Q from target, K/V from context).\n      2. Self-Attention   (standard, over target tokens only).\n      3. Feed-Forward MLP.\n\n    All sub-layers use pre-norm convention (LayerNorm before each sublayer).\n\n    Args:\n        dim        : Token embedding dimension (decoder_dim).\n        num_heads  : Number of attention heads.\n        mlp_ratio  : Hidden-to-dim ratio for the MLP.\n        dropout    : Dropout probability for attention weights.\n    """\n\n    def __init__(\n        self,\n        dim: int,\n        num_heads: int,\n        mlp_ratio: float = 4.0,\n        dropout: float = 0.0,\n    ) -> None:\n        super().__init__()\n        assert dim % num_heads == 0, (\n            f"dim={dim} must be divisible by num_heads={num_heads}."\n        )\n\n        # --- Cross-attention ---\n        self.norm_q  = nn.LayerNorm(dim)\n        self.norm_kv = nn.LayerNorm(dim)\n        self.cross_attn = nn.MultiheadAttention(\n            embed_dim=dim,\n            num_heads=num_heads,\n            dropout=dropout,\n            batch_first=True,\n        )\n\n        # --- Self-attention ---\n        self.norm_sa = nn.LayerNorm(dim)\n        self.self_attn = nn.MultiheadAttention(\n            embed_dim=dim,\n            num_heads=num_heads,\n            dropout=dropout,\n            batch_first=True,\n        )\n\n        # --- MLP ---\n        hidden_dim = int(dim * mlp_ratio)\n        self.norm_mlp = nn.LayerNorm(dim)\n        self.mlp = nn.Sequential(\n            nn.Linear(dim, hidden_dim),\n            nn.GELU(),\n            nn.Linear(hidden_dim, dim),\n        )\n\n        self.drop = nn.Dropout(dropout)\n\n    def forward(\n        self,\n        tgt: torch.Tensor,\n        ctx: torch.Tensor,\n    ) -> torch.Tensor:\n        """\n        Args:\n            tgt : (B, N_tgt, dim) -- query tokens (masked target patches).\n            ctx : (B, N_ctx, dim) -- key/value tokens (context-frame encoder output).\n\n        Returns:\n            tgt : (B, N_tgt, dim) -- updated target tokens.\n        """\n        # 1. Cross-attention\n        q  = self.norm_q(tgt)\n        kv = self.norm_kv(ctx)\n        ca_out, _ = self.cross_attn(query=q, key=kv, value=kv)\n        tgt = tgt + self.drop(ca_out)\n\n        # 2. Self-attention\n        sa_in = self.norm_sa(tgt)\n        sa_out, _ = self.self_attn(query=sa_in, key=sa_in, value=sa_in)\n        tgt = tgt + self.drop(sa_out)\n\n        # 3. MLP\n        tgt = tgt + self.drop(self.mlp(self.norm_mlp(tgt)))\n        return tgt\n\n\n# ---------------------------------------------------------------------------\n# CrossFrameDecoder\n# ---------------------------------------------------------------------------\n\nclass CrossFrameDecoder(nn.Module):\n    """\n    SiamMAE-style cross-frame reconstruction decoder.\n\n    Given:\n      - ctx_tokens : encoder output for Frame-i  (B, N_ctx, embed_dim)\n                     (all encoder tokens, optionally including CLS).\n      - mask       : binary mask (B, L), 1 = masked, 0 = visible.\n      - delta_t    : scalar time gap per sample (B, 1), normalised to [0, 1].\n\n    Returns:\n      - pred : (B, L, patch_size^2 * in_chans) -- predictions for ALL patches.\n               Callers compute MSE loss only on masked indices.\n\n    Internal flow:\n      1. Project ctx_tokens: embed_dim -> decoder_dim.\n      2. Build query tensor: mask_token (at masked positions) + pos_embed + Delta-t.\n      3. Run ``depth`` CrossAttentionBlocks (Q=queries, K/V=ctx).\n      4. LayerNorm + linear head -> pixel predictions.\n\n    Args:\n        embed_dim    : Encoder output dimension (ViC-MAE backbone embed_dim).\n        decoder_dim  : Internal decoder token dimension.\n        depth        : Number of CrossAttentionBlocks.\n        num_heads    : Attention heads per block.\n        patch_size   : Spatial patch size in pixels.\n        in_chans     : Number of image channels.\n        img_size     : Input image spatial size (square assumed).\n        mlp_ratio    : MLP ratio inside each CrossAttentionBlock.\n        dropout      : Dropout for attention layers.\n    """\n\n    def __init__(\n        self,\n        embed_dim: int = 1024,\n        decoder_dim: int = 512,\n        depth: int = 4,\n        num_heads: int = 8,\n        patch_size: int = 16,\n        in_chans: int = 3,\n        img_size: int = 224,\n        mlp_ratio: float = 4.0,\n        dropout: float = 0.0,\n    ) -> None:\n        super().__init__()\n\n        self.decoder_dim  = decoder_dim\n        self.patch_size   = patch_size\n        self.in_chans     = in_chans\n        self.num_patches  = (img_size // patch_size) ** 2\n        self.grid_size    = img_size // patch_size\n\n        # --- Context projection ---\n        self.ctx_proj = nn.Linear(embed_dim, decoder_dim, bias=True)\n        self.proj = self.ctx_proj  # alias for convenience\n\n        # --- Learnable mask token ---\n        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))\n        nn.init.normal_(self.mask_token, std=0.02)\n\n        # --- Spatial positional embedding (fixed 2-D sin-cos, covers all L patches) ---\n        pos = _get_2d_sincos_pos_embed(decoder_dim, self.grid_size)  # (L, D)\n        self.register_buffer("pos_embed", pos.unsqueeze(0))           # (1, L, D)\n\n        # --- Continuous time-gap embedding Delta-t ---\n        # 2-layer MLP: R^1 -> decoder_dim, using SiLU activation.\n        self.time_embed = nn.Sequential(\n            nn.Linear(1, decoder_dim),\n            nn.SiLU(),\n            nn.Linear(decoder_dim, decoder_dim),\n        )\n        self.time_mlp = self.time_embed  # alias for backward-compatibility\n\n        # --- Cross-attention blocks ---\n        self.blocks = nn.ModuleList([\n            CrossAttentionBlock(\n                dim=decoder_dim,\n                num_heads=num_heads,\n                mlp_ratio=mlp_ratio,\n                dropout=dropout,\n            )\n            for _ in range(depth)\n        ])\n\n        # --- Output head ---\n        self.norm = nn.LayerNorm(decoder_dim)\n        self.pred = nn.Linear(\n            decoder_dim,\n            patch_size * patch_size * in_chans,\n            bias=True,\n        )\n\n        self._init_weights()\n\n    # ------------------------------------------------------------------\n    # Weight initialisation\n    # ------------------------------------------------------------------\n\n    def _init_weights(self) -> None:\n        """Initialise Linear and LayerNorm weights following MAE conventions."""\n        for m in self.modules():\n            if isinstance(m, nn.Linear):\n                nn.init.xavier_uniform_(m.weight)\n                if m.bias is not None:\n                    nn.init.constant_(m.bias, 0.0)\n            elif isinstance(m, nn.LayerNorm):\n                nn.init.constant_(m.weight, 1.0)\n                nn.init.constant_(m.bias, 0.0)\n        # Mask token re-init (already in __init__, kept for explicit visibility)\n        nn.init.normal_(self.mask_token, std=0.02)\n\n    # ------------------------------------------------------------------\n    # Forward\n    # ------------------------------------------------------------------\n\n    def forward(\n        self,\n        ctx_tokens: torch.Tensor,\n        mask_or_delta: Optional[torch.Tensor] = None,\n        delta_t: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor:\n        """\n        Args:\n            ctx_tokens    : (B, N_ctx, embed_dim) -- Frame-i encoder tokens.\n            mask_or_delta : (B, L) binary mask, OR (B, 1) delta_t if mask omitted.\n            delta_t       : (B, 1) normalised time gap |t_j - t_i| in [0, 1].\n\n        Returns:\n            pred : (B, L, patch_size^2 * in_chans).\n        """\n        B = ctx_tokens.shape[0]\n\n        # Handle 2-arg calling: forward(ctx_tokens, delta_t)\n        if delta_t is None and mask_or_delta is not None:\n            if mask_or_delta.ndim == 2 and mask_or_delta.shape[-1] == 1:\n                delta_t = mask_or_delta\n                mask = torch.ones(B, self.num_patches, device=ctx_tokens.device, dtype=torch.float32)\n            else:\n                mask = mask_or_delta\n                delta_t = torch.zeros(B, 1, device=ctx_tokens.device, dtype=torch.float32)\n        elif mask_or_delta is None and delta_t is None:\n            mask = torch.ones(B, self.num_patches, device=ctx_tokens.device, dtype=torch.float32)\n            delta_t = torch.zeros(B, 1, device=ctx_tokens.device, dtype=torch.float32)\n        else:\n            mask = mask_or_delta\n            if delta_t is None:\n                delta_t = torch.zeros(B, 1, device=ctx_tokens.device, dtype=torch.float32)\n\n        B, L = mask.shape\n        assert L == self.num_patches, (\n            f"Expected {self.num_patches} patches, got {L}."\n        )\n\n        # 1. Project context tokens into decoder dim.\n        ctx = self.ctx_proj(ctx_tokens)                    # (B, N_ctx, D)\n\n        # 2. Time-gap embedding: (B, 1) -> (B, D) -> (B, 1, D)\n        dt_embed = self.time_mlp(delta_t.float())          # (B, D)\n        dt_embed = dt_embed.unsqueeze(1)                   # (B, 1, D)\n\n        # 3. Build query tokens for ALL L patch positions.\n        #    SiamMAE style: decoder does NOT see target visible patches.\n        #    Masked positions  -> mask_token + pos + Delta-t\n        #    Visible positions -> zeros      + pos + Delta-t\n        pos = self.pos_embed.expand(B, -1, -1)             # (B, L, D)\n\n        queries = torch.zeros(\n            B, L, self.decoder_dim,\n            device=ctx_tokens.device,\n            dtype=ctx_tokens.dtype,\n        )\n\n        # Inject learnable mask token at masked positions\n        mask_expanded = mask.bool().unsqueeze(-1)          # (B, L, 1)\n        mask_tok      = self.mask_token.expand(B, L, -1)   # (B, L, D)\n        queries = torch.where(mask_expanded, mask_tok, queries)\n\n        # Add positional and temporal embeddings to ALL query positions\n        queries = queries + pos + dt_embed                 # (B, L, D)\n\n        # 4. Run cross-attention stack.\n        for blk in self.blocks:\n            queries = blk(tgt=queries, ctx=ctx)\n\n        # 5. Output head.\n        queries = self.norm(queries)                       # (B, L, D)\n        pred    = self.pred(queries)                       # (B, L, p*p*C)\n\n        return pred\n',
    os.path.join(REPO_DIR, "models", "extended_vicmae.py"): '"""\nmodels/extended_vicmae.py\n--------------------------\nExtended ViC-MAE: wraps the base ViCMAE model with the SiamMAE-style\nCrossFrameDecoder and provides:\n\n  - forward_cross_recon()     -- cross-frame masked reconstruction loss\n  - compute_preservation_loss() -- cosine-distance feature preservation\n\nThis module is the central interface for the Stage-1 / Stage-2 training\nprotocol described in the ViC-MAE temporal extension specification.\n"""\n\nfrom __future__ import annotations\n\nimport sys\nimport os\nfrom copy import deepcopy\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n# Allow imports from parent directory when running standalone\n_PARENT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nif _PARENT not in sys.path:\n    sys.path.insert(0, _PARENT)\n\nfrom models_vicmae import ViCMAE\nfrom models.cross_decoder import CrossFrameDecoder\n\n\n# ---------------------------------------------------------------------------\n# LoRA helpers (lightweight, no external dependency)\n# ---------------------------------------------------------------------------\n\nclass LoRALinear(nn.Module):\n    """\n    Low-Rank Adaptation layer wrapping an existing nn.Linear.\n\n    Adds trainable low-rank matrices A (in -> r) and B (r -> out) whose\n    product alpha/r * B @ A is added to the frozen base weight output.\n\n    Args:\n        linear : Frozen nn.Linear module to wrap.\n        r      : LoRA rank.\n        alpha  : LoRA scaling factor. Effective scale = alpha / r.\n    """\n\n    def __init__(self, linear: nn.Linear, r: int = 16, alpha: float = 32.0) -> None:\n        super().__init__()\n        self.linear = linear\n        self.r      = r\n        self.scale  = alpha / r\n\n        # Freeze base weight\n        for p in self.linear.parameters():\n            p.requires_grad_(False)\n\n        in_features  = linear.in_features\n        out_features = linear.out_features\n        dev = linear.weight.device\n        dtype = linear.weight.dtype\n        self.lora_A  = nn.Parameter(torch.randn(r, in_features, device=dev, dtype=dtype) * 0.01)\n        self.lora_B  = nn.Parameter(torch.zeros(out_features, r, device=dev, dtype=dtype))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        base_out  = self.linear(x)\n        lora_out  = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale\n        return base_out + lora_out\n\n\ndef inject_lora(module: nn.Module, r: int = 16, alpha: float = 32.0) -> nn.Module:\n    """\n    Replace every ``nn.Linear`` inside ``module`` with a ``LoRALinear``,\n    freezing the base weights and adding trainable low-rank adapters.\n\n    Returns the modified module in-place.\n    """\n    for name, child in list(module.named_children()):\n        if isinstance(child, nn.Linear):\n            setattr(module, name, LoRALinear(child, r=r, alpha=alpha))\n        else:\n            inject_lora(child, r=r, alpha=alpha)\n    return module\n\n\n# ---------------------------------------------------------------------------\n# ExtendedViCMAE\n# ---------------------------------------------------------------------------\n\nclass ExtendedViCMAE(nn.Module):\n    """\n    Temporal extension of ViC-MAE combining:\n\n      1. Base ViCMAE losses   : L_mae  (reconstruction) + L_nce (contrastive).\n      2. Cross-frame loss     : L_cross (SiamMAE-style masked reconstruction of\n                                Frame-j conditioned on Frame-i tokens + Delta-t).\n      3. Preservation loss    : L_pres (cosine distance between active encoder\n                                patch tokens and a frozen teacher\'s tokens on\n                                image-only batches).\n\n    Training stages\n    ---------------\n    Stage 1  -- decoder warm-up:\n        call ``freeze_backbone()``; only cross_decoder trains (lr=1e-4).\n    Stage 2  -- joint fine-tuning:\n        call ``unfreeze_top_blocks(n=4)`` OR ``inject_lora_adapters()``;\n        encoder uses smaller lr than cross_decoder.\n\n    Args:\n        vicmae_cfg      : kwargs forwarded to ViCMAE.__init__.\n        decoder_dim     : Inner dimension of CrossFrameDecoder.\n        decoder_depth   : Number of CrossAttentionBlocks.\n        decoder_heads   : Attention heads per CrossAttentionBlock.\n        img_size        : Image spatial size (square).\n        cross_mask_ratio: Fraction of Frame-j patches to mask (default 0.95).\n    """\n\n    def __init__(\n        self,\n        vicmae_cfg: Optional[Dict] = None,\n        decoder_dim: int = 512,\n        decoder_depth: int = 4,\n        decoder_heads: int = 8,\n        img_size: int = 224,\n        cross_mask_ratio: float = 0.95,\n    ) -> None:\n        super().__init__()\n\n        # --- Base ViC-MAE ---\n        if isinstance(vicmae_cfg, nn.Module):\n            self.vicmae = vicmae_cfg\n        elif isinstance(vicmae_cfg, dict):\n            self.vicmae = ViCMAE(**vicmae_cfg)\n        elif vicmae_cfg is None:\n            self.vicmae = ViCMAE()\n        else:\n            self.vicmae = ViCMAE()\n\n        embed_dim  = self.vicmae.patch_embed.proj.out_channels  # encoder dim\n        patch_size = self.vicmae.patch_embed.patch_size[0]\n        in_chans   = self.vicmae.patch_embed.proj.in_channels\n\n        # --- Cross-frame decoder ---\n        self.cross_decoder = CrossFrameDecoder(\n            embed_dim=embed_dim,\n            decoder_dim=decoder_dim,\n            depth=decoder_depth,\n            num_heads=decoder_heads,\n            patch_size=patch_size,\n            in_chans=in_chans,\n            img_size=img_size,\n        )\n\n        self.cross_mask_ratio = cross_mask_ratio\n        self._patch_size      = patch_size\n        self._in_chans        = in_chans\n\n    # ------------------------------------------------------------------\n    # Stage control helpers\n    # ------------------------------------------------------------------\n\n    def freeze_backbone(self) -> None:\n        """Stage 1: freeze the entire ViCMAE encoder + decoder."""\n        self.vicmae.requires_grad_(False)\n\n    def unfreeze_backbone(self) -> None:\n        """Unfreeze entire ViCMAE (used before Stage 2 full fine-tune)."""\n        self.vicmae.requires_grad_(True)\n\n    def unfreeze_top_blocks(self, n: int = 4) -> None:\n        """\n        Stage 2 option A: keep backbone mostly frozen; unfreeze only the\n        top-``n`` Transformer blocks and the final norm layer.\n        """\n        self.vicmae.requires_grad_(False)\n        total = len(self.vicmae.blocks)\n        for blk in self.vicmae.blocks[total - n:]:\n            blk.requires_grad_(True)\n        self.vicmae.norm.requires_grad_(True)\n\n    def inject_lora_adapters(self, r: int = 16, alpha: float = 32.0) -> None:\n        """\n        Stage 2 option B: inject LoRA adapters into all Linear layers of the\n        ViCMAE encoder; freeze base weights, only adapters are trainable.\n        """\n        device = next(self.vicmae.parameters()).device\n        self.vicmae.requires_grad_(False)\n        inject_lora(self.vicmae.blocks, r=r, alpha=alpha)\n        self.to(device)\n\n    def trainable_parameters(self) -> list:\n        """Return list of all parameters with requires_grad=True."""\n        return [p for p in self.parameters() if p.requires_grad]\n\n    # ------------------------------------------------------------------\n    # Patchification helpers\n    # ------------------------------------------------------------------\n\n    def _patchify(self, imgs: torch.Tensor) -> torch.Tensor:\n        """\n        Decompose images into flattened patches (normalised per-patch).\n\n        Args:\n            imgs : (B, C, H, W)\n\n        Returns:\n            patches : (B, L, patch_size^2 * C), zero-mean unit-var per patch.\n        """\n        p = self._patch_size\n        B, C, H, W = imgs.shape\n        assert H % p == 0 and W % p == 0\n        h, w = H // p, W // p\n\n        x = imgs.reshape(B, C, h, p, w, p)\n        x = torch.einsum("nchpwq->nhwpqc", x)\n        x = x.reshape(B, h * w, p * p * C)\n\n        # Per-patch normalisation (zero mean, unit variance)\n        mean = x.mean(dim=-1, keepdim=True)\n        var  = x.var(dim=-1, keepdim=True)\n        x    = (x - mean) / (var + 1e-6).sqrt()\n        return x\n\n    def _generate_mask(\n        self,\n        B: int,\n        L: int,\n        mask_ratio: float,\n        device: torch.device,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        """\n        Generate random binary mask.\n\n        Returns:\n            mask        : (B, L) float tensor; 1 = masked, 0 = visible.\n            ids_restore : (B, L) permutation to unshuffle.\n        """\n        len_keep   = max(1, int(L * (1.0 - mask_ratio)))\n        noise      = torch.rand(B, L, device=device)\n        ids_shuffle = torch.argsort(noise, dim=1)\n        ids_restore = torch.argsort(ids_shuffle, dim=1)\n\n        mask          = torch.ones(B, L, device=device)\n        mask[:, :len_keep] = 0.0\n        mask          = torch.gather(mask, dim=1, index=ids_restore)\n        return mask, ids_restore\n\n    # ------------------------------------------------------------------\n    # Cross-frame reconstruction\n    # ------------------------------------------------------------------\n\n    def forward_cross_recon(\n        self,\n        ctx_tokens: torch.Tensor,\n        target_imgs: torch.Tensor,\n        delta_t: torch.Tensor,\n        mask_ratio: float = 0.95,\n    ) -> torch.Tensor:\n        """\n        SiamMAE-style cross-frame reconstruction loss.\n\n        Procedure:\n          1. Patchify and per-patch normalise target_imgs.\n          2. Generate heavy mask (mask_ratio ~ 0.95) over target patches.\n          3. Run CrossFrameDecoder: queries <- mask_token + pos + Delta-t,\n             K/V <- ctx_tokens (Frame-i encoder output).\n          4. Compute MSE loss only on masked patch indices.\n\n        Args:\n            ctx_tokens  : (B, N_ctx, embed_dim) -- Frame-i encoder tokens\n                          (output of vicmae.forward_encoder, incl. CLS).\n            target_imgs : (B, C, H, W) -- raw Frame-j images (not patchified).\n            delta_t     : (B, 1) -- normalised time gap |t_j - t_i|.\n            mask_ratio  : Fraction of Frame-j patches to mask (default 0.95).\n\n        Returns:\n            loss_cross : scalar MSE loss on masked patches.\n        """\n        B  = target_imgs.shape[0]\n        L  = self.cross_decoder.num_patches\n\n        # 1. Ground-truth patches (normalised)\n        target = self._patchify(target_imgs)          # (B, L, p*p*C)\n\n        # 2. Generate mask\n        mask, _ = self._generate_mask(B, L, mask_ratio, target_imgs.device)\n\n        # 3. Decode\n        pred = self.cross_decoder(ctx_tokens, mask, delta_t)  # (B, L, p*p*C)\n\n        # 4. MSE on masked patches only\n        loss_per_patch = ((pred - target) ** 2).mean(dim=-1)  # (B, L)\n        loss_cross     = (loss_per_patch * mask).sum() / mask.sum().clamp(min=1)\n\n        return loss_cross, pred, mask\n\n    # ------------------------------------------------------------------\n    # Feature preservation loss\n    # ------------------------------------------------------------------\n\n    @staticmethod\n    def compute_preservation_loss(\n        current_tokens: torch.Tensor,\n        frozen_tokens: torch.Tensor,\n    ) -> torch.Tensor:\n        """\n        Cosine-distance preservation loss between active and frozen encoder tokens.\n        """\n        if current_tokens.shape[1] == frozen_tokens.shape[1] + 1:\n            cur = current_tokens[:, 1:, :]\n            frz = frozen_tokens\n        elif frozen_tokens.shape[1] == current_tokens.shape[1] + 1:\n            cur = current_tokens\n            frz = frozen_tokens[:, 1:, :]\n        else:\n            cur = current_tokens\n            frz = frozen_tokens\n\n        # Normalise along feature dimension\n        cur_norm = F.normalize(cur, dim=-1)\n        frz_norm = F.normalize(frz, dim=-1)\n\n        # Cosine similarity per patch, averaged over batch and patches\n        cos_sim   = (cur_norm * frz_norm).sum(dim=-1)   # (B, L)\n        loss_pres = 1.0 - cos_sim.mean()\n        return loss_pres\n\n    preservation_loss = compute_preservation_loss  # alias\n\n    # ------------------------------------------------------------------\n    # Convenience: get encoder tokens without masking\n    # ------------------------------------------------------------------\n\n    def encode_full(self, imgs: torch.Tensor) -> torch.Tensor:\n        """Run ViCMAE encoder on the FULL (unmasked) image."""\n        tokens, _, _ = self.vicmae.forward_encoder(imgs, mask_ratio=0.0)\n        return tokens\n\n    def encode_frame(self, imgs: torch.Tensor, mask_ratio: float = 0.0):\n        """Encode a frame with the given mask ratio."""\n        return self.vicmae.forward_encoder(imgs, mask_ratio=mask_ratio)\n\n    def forward_mae(self, imgs: torch.Tensor, mask_ratio: float = 0.5):\n        """Forward pass for standard MAE reconstruction."""\n        latent, mask, ids_restore = self.vicmae.forward_encoder(imgs, mask_ratio)\n        pred = self.vicmae.forward_decoder(latent, ids_restore)\n        loss = self.vicmae.mse_loss(imgs, pred, mask)\n        return loss, pred, mask\n\n    def forward(self, imgs):\n        if isinstance(imgs, dict):\n            if imgs.get("is_video", False):\n                frame_i = imgs["frame_i"]\n                frame_j = imgs["frame_j"]\n                delta_t = imgs["delta_t"]\n                mask_ratio = imgs.get("mask_ratio", 0.5)\n                lambda_nce = imgs.get("lambda_nce", 0.1)\n                lambda_cross = imgs.get("lambda_cross", 1.0)\n\n                l_mae_i, _, _ = self.forward_mae(frame_i, mask_ratio)\n                l_mae_j, _, _ = self.forward_mae(frame_j, mask_ratio)\n\n                enc_i, _, _ = self.vicmae.forward_encoder(frame_i, mask_ratio)\n                enc_j, _, _ = self.vicmae.forward_encoder(frame_j, mask_ratio)\n                pool_i = self.vicmae.projector(enc_i[:, 1:, :].mean(dim=1))\n                pool_j = self.vicmae.projector(enc_j[:, 1:, :].mean(dim=1))\n                z = torch.cat([pool_i, pool_j], dim=0)\n                l_nce = self.vicmae.info_nce_loss(z, self.vicmae.temperature)\n\n                loss_cross, _, _ = self.forward_cross_recon(\n                    enc_i[:, 1:, :], frame_j, delta_t, mask_ratio=self.cross_mask_ratio\n                )\n\n                total = l_mae_i + l_mae_j + lambda_nce * l_nce + lambda_cross * loss_cross\n                return {\n                    "loss": total,\n                    "loss_mae_i": l_mae_i,\n                    "loss_mae_j": l_mae_j,\n                    "loss_nce": l_nce,\n                    "loss_cross": loss_cross,\n                }\n            else:\n                image = imgs["image"]\n                mask_ratio = imgs.get("mask_ratio", 0.5)\n                lambda_pres = imgs.get("lambda_pres", 0.5)\n                frozen_tokens = imgs["frozen_tokens"]\n\n                l_mae, _, _ = self.forward_mae(image, mask_ratio)\n                act_tokens = self.encode_full(image)[:, 1:, :]\n                l_pres = self.compute_preservation_loss(act_tokens, frozen_tokens)\n\n                total = l_mae + lambda_pres * l_pres\n                return {\n                    "loss": total,\n                    "loss_mae": l_mae,\n                    "loss_pres": l_pres,\n                }\n\n        return self.vicmae(imgs)\n',
    os.path.join(REPO_DIR, "models", "lora.py"): '"""Minimal LoRA (Low-Rank Adaptation) utilities for the ViT backbone.\n\nSelf-contained implementation (no external LoRA dependency): wraps selected\n``nn.Linear`` layers with a frozen base weight plus a trainable low-rank\nupdate ``B @ A`` scaled by ``alpha / r``. Used by Stage 2 of the\ncontinued-training protocol to unfreeze the backbone with minimal\nparameter overhead (r=16, alpha=32 by default).\n"""\nfrom __future__ import annotations\n\nimport math\nfrom typing import Iterable, List\n\nimport torch\nimport torch.nn as nn\n\n\nclass LoRALinear(nn.Module):\n    """A frozen ``nn.Linear`` plus a trainable low-rank residual.\n\n    ``forward(x) = base(x) + (x @ A^T @ B^T) * (alpha / r)``\n\n    ``B`` is zero-initialized so the wrapped layer is exactly equivalent to\n    the base linear at initialization.\n    """\n\n    def __init__(self, base: nn.Linear, r: int = 16, alpha: float = 32.0) -> None:\n        super().__init__()\n        if r <= 0:\n            raise ValueError(f"LoRA rank r must be positive, got {r}")\n        self.base = base\n        self.r = r\n        self.alpha = alpha\n        self.scaling = alpha / r\n\n        dev = base.weight.device\n        dtype = base.weight.dtype\n        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features, device=dev, dtype=dtype))\n        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=dev, dtype=dtype))\n        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))\n        # lora_B stays zero -> identity residual at init.\n\n        # The base weight is frozen; only the LoRA factors are trainable.\n        self.base.weight.requires_grad_(False)\n        if self.base.bias is not None:\n            self.base.bias.requires_grad_(False)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        out = self.base(x)\n        if x.ndim == 2:\n            lora = (x @ self.lora_A.t()) @ self.lora_B.t()\n        else:  # [B, L, D] token sequences\n            shape = x.shape\n            lora = (x.reshape(-1, shape[-1]) @ self.lora_A.t()) @ self.lora_B.t()\n            lora = lora.reshape(*shape[:-1], self.base.out_features)\n        return out + lora * self.scaling\n\n    def merged_weight(self) -> torch.Tensor:\n        """Return the effective weight (base + low-rank update) for export."""\n        return self.base.weight + (self.lora_B @ self.lora_A) * self.scaling\n\n\ndef inject_lora(\n    model: nn.Module,\n    target_modules: Iterable[str] = ("qkv", "proj"),\n    r: int = 16,\n    alpha: float = 32.0,\n) -> List[str]:\n    """Replace matching ``nn.Linear`` submodules with :class:`LoRALinear`.\n\n    Args:\n        model:          the module to modify in place (e.g. the ViT backbone).\n        target_modules: attribute names of ``nn.Linear`` children to wrap.\n        r:              LoRA rank.\n        alpha:          LoRA scaling factor (effective scale = alpha / r).\n\n    Returns:\n        List of fully-qualified names of the wrapped parameters.\n    """\n    target_set = set(target_modules)\n    wrapped: List[str] = []\n\n    def _recurse(parent: nn.Module, prefix: str) -> None:\n        for name, child in list(parent.named_children()):\n            full_name = f"{prefix}.{name}" if prefix else name\n            if isinstance(child, nn.Linear) and name in target_set:\n                setattr(parent, name, LoRALinear(child, r=r, alpha=alpha))\n                wrapped.append(full_name)\n            else:\n                _recurse(child, full_name)\n\n    _recurse(model, "")\n    return wrapped\n\n\ndef mark_only_lora_trainable(model: nn.Module) -> int:\n    """Freeze every parameter except LoRA factors. Returns the trainable count."""\n    for p in model.parameters():\n        p.requires_grad_(False)\n    n_trainable = 0\n    for module in model.modules():\n        if isinstance(module, LoRALinear):\n            module.lora_A.requires_grad_(True)\n            module.lora_B.requires_grad_(True)\n            n_trainable += module.lora_A.numel() + module.lora_B.numel()\n    return n_trainable\n\n\ndef lora_state_dict(model: nn.Module) -> dict:\n    """Return only the LoRA parameters (for compact checkpointing)."""\n    return {\n        k: v.detach().cpu()\n        for k, v in model.state_dict().items()\n        if "lora_A" in k or "lora_B" in k\n    }\n',
    os.path.join(REPO_DIR, "datasets", "__init__.py"): "",
    os.path.join(REPO_DIR, "datasets", "video_image_dataset.py"): '"""\ndatasets/video_image_dataset.py\n--------------------------------\nMixed video-frame / static-image dataset for ViC-MAE temporal extension.\n\nEach sample yields a dictionary with a stable schema so that the training\nloop can transparently handle both modalities:\n\n  Video sample (is_video=True):\n    {\n        "frame_i"  : Tensor[C, H, W],   # context frame\n        "frame_j"  : Tensor[C, H, W],   # target  frame\n        "delta_t"  : Tensor[1],          # normalised |t_j - t_i| in [0, 1]\n        "is_video" : True,\n    }\n\n  Image sample (is_video=False):\n    {\n        "image"    : Tensor[C, H, W],\n        "is_video" : False,\n    }\n\nThe two datasets can be merged via MixedDataset, which interleaves\nsamples from both sources according to a configurable video_ratio.\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport random\nfrom pathlib import Path\nfrom typing import Callable, List, Optional, Sequence, Tuple\n\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nimport torchvision.transforms as T\nfrom PIL import Image\n\n\ntry:\n    import cv2\n    _HAS_CV2 = True\nexcept ImportError:\n    _HAS_CV2 = False\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD  = (0.229, 0.224, 0.225)\n\nVIDEO_EXTENSIONS = (".mp4", ".avi", ".mov", ".mkv", ".webm")\nIMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")\n\n\ndef extract_video_frames(\n    video_path: str | Path,\n    output_dir: str | Path,\n    max_fps: Optional[int] = 15,\n) -> int:\n    """\n    Extract frames from a raw video file into output_dir as JPG files.\n\n    Args:\n        video_path : Path to video file (.mp4, .avi, etc.).\n        output_dir : Output directory for extracted frames.\n        max_fps    : Target sampling FPS (None for all frames).\n\n    Returns:\n        Number of frames saved.\n    """\n    if not _HAS_CV2:\n        raise ImportError("opencv-python (cv2) is required to extract video frames.")\n    output_path = Path(output_dir)\n    output_path.mkdir(parents=True, exist_ok=True)\n    cap = cv2.VideoCapture(str(video_path))\n    if not cap.isOpened():\n        raise ValueError(f"Could not open video file: {video_path}")\n\n    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0\n    step = max(1, int(round(fps / max_fps))) if max_fps else 1\n\n    frame_idx = 0\n    saved_count = 0\n    while True:\n        ret, frame = cap.read()\n        if not ret or frame is None:\n            break\n        if frame_idx % step == 0:\n            out_file = output_path / f"{saved_count:06d}.jpg"\n            cv2.imwrite(str(out_file), frame)\n            saved_count += 1\n        frame_idx += 1\n    cap.release()\n    return saved_count\n\n\ndef default_frame_transform(img_size: int = 224) -> T.Compose:\n    return T.Compose([\n        T.RandomResizedCrop(img_size, scale=(0.5, 1.0)),\n        T.RandomHorizontalFlip(),\n        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),\n        T.RandomGrayscale(p=0.2),\n        T.ToTensor(),\n        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),\n    ])\n\n\ndef default_image_transform(img_size: int = 224) -> T.Compose:\n    return T.Compose([\n        T.RandomResizedCrop(img_size, scale=(0.5, 1.0)),\n        T.RandomHorizontalFlip(),\n        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),\n        T.RandomGrayscale(p=0.2),\n        T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),\n        T.ToTensor(),\n        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),\n    ])\n\n\nclass VideoFramePairDataset(Dataset):\n    """\n    Loads pairs of frames from pre-extracted clip directories OR directly\n    from raw video files (.mp4, .avi, .mov, etc.).\n\n    Directory layout option 1 (Image clips):\n        root/\n          clip_001/\n            000000.jpg\n            000001.jpg\n            ...\n\n    Directory layout option 2 (Raw video files):\n        root/\n          video_001.mp4\n          video_002.mp4\n          ...\n        OR a single video file passed as `root`.\n\n    Args:\n        root          : Root directory containing clip sub-folders or video files.\n        max_frame_gap : Maximum frame-index separation; delta_t normalised by this.\n        transform     : Optional transform applied to each frame.\n        img_size      : Spatial size for default transform.\n        extensions    : Valid image extensions for clip sub-folders.\n        video_extensions: Valid raw video extensions.\n    """\n\n    def __init__(\n        self,\n        root: str,\n        max_frame_gap: int = 48,\n        transform: Optional[Callable] = None,\n        img_size: int = 224,\n        extensions: Tuple[str, ...] = IMAGE_EXTENSIONS,\n        video_extensions: Tuple[str, ...] = VIDEO_EXTENSIONS,\n    ) -> None:\n        super().__init__()\n        self.root          = Path(root)\n        self.max_frame_gap = max_frame_gap\n        self.img_size      = img_size\n        self.transform     = transform or default_frame_transform(img_size)\n\n        self.clips: List[List[Path]] = []\n        self.video_files: List[Path] = []\n\n        if self.root.is_file() and self.root.suffix.lower() in video_extensions:\n            self.video_files = [self.root]\n        elif self.root.is_dir():\n            # Check for direct video files first\n            self.video_files = sorted([\n                p for p in self.root.rglob("*")\n                if p.is_file() and p.suffix.lower() in video_extensions\n            ])\n            # Check for directories containing image frames\n            for clip_dir in sorted(self.root.iterdir()):\n                if not clip_dir.is_dir():\n                    continue\n                frames = sorted([\n                    p for p in clip_dir.iterdir()\n                    if p.suffix.lower() in extensions\n                ])\n                if len(frames) >= 2:\n                    self.clips.append(frames)\n\n        if not self.clips and not self.video_files:\n            raise RuntimeError(\n                f"No valid video clips or video files found under {root}. "\n                f"Supported: folders of frames ({extensions}) or video files ({video_extensions})."\n            )\n\n    def __len__(self) -> int:\n        return len(self.video_files) if self.video_files else len(self.clips)\n\n    def __getitem__(self, idx: int) -> dict:\n        if self.video_files:\n            if not _HAS_CV2:\n                raise ImportError("opencv-python (cv2) is required to read raw video files.")\n            vpath = self.video_files[idx % len(self.video_files)]\n            cap = cv2.VideoCapture(str(vpath))\n            n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n            if n >= 2:\n                gap = random.randint(1, min(self.max_frame_gap, n - 1))\n                i_idx = random.randint(0, n - 1 - gap)\n                j_idx = i_idx + gap\n                cap.set(cv2.CAP_PROP_POS_FRAMES, i_idx)\n                ret_i, f_i = cap.read()\n                cap.set(cv2.CAP_PROP_POS_FRAMES, j_idx)\n                ret_j, f_j = cap.read()\n            else:\n                ret_i, ret_j, f_i, f_j, gap = False, False, None, None, 1\n            cap.release()\n\n            if ret_i and ret_j and f_i is not None and f_j is not None:\n                img_i = Image.fromarray(cv2.cvtColor(f_i, cv2.COLOR_BGR2RGB))\n                img_j = Image.fromarray(cv2.cvtColor(f_j, cv2.COLOR_BGR2RGB))\n            else:\n                img_i = Image.new("RGB", (self.img_size, self.img_size), color=(128, 128, 128))\n                img_j = Image.new("RGB", (self.img_size, self.img_size), color=(128, 128, 128))\n\n            frame_i = self.transform(img_i)\n            frame_j = self.transform(img_j)\n            delta_t = torch.tensor([gap / self.max_frame_gap], dtype=torch.float32)\n            return {"frame_i": frame_i, "frame_j": frame_j, "delta_t": delta_t, "is_video": True}\n\n        frames = self.clips[idx]\n        n      = len(frames)\n        gap    = random.randint(1, min(self.max_frame_gap, n - 1))\n        i_idx  = random.randint(0, n - 1 - gap)\n        j_idx  = i_idx + gap\n\n        frame_i = self.transform(Image.open(frames[i_idx]).convert("RGB"))\n        frame_j = self.transform(Image.open(frames[j_idx]).convert("RGB"))\n        delta_t = torch.tensor([gap / self.max_frame_gap], dtype=torch.float32)\n\n        return {"frame_i": frame_i, "frame_j": frame_j,\n                "delta_t": delta_t, "is_video": True}\n\n\nclass StaticImageDataset(Dataset):\n    """\n    Flat / class-organised image dataset.\n\n    Args:\n        root      : Root directory; images discovered recursively.\n        transform : Optional transform applied to each image.\n        img_size  : Spatial size for default transform.\n        extensions: Valid file extensions.\n    """\n\n    def __init__(\n        self,\n        root: str,\n        transform: Optional[Callable] = None,\n        img_size: int = 224,\n        extensions: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".JPEG"),\n    ) -> None:\n        super().__init__()\n        self.transform = transform or default_image_transform(img_size)\n        self.paths: List[Path] = sorted([\n            p for p in Path(root).rglob("*")\n            if p.is_file() and p.suffix.lower() in {e.lower() for e in extensions}\n        ])\n        if not self.paths:\n            raise RuntimeError(f"No images found under {root}.")\n\n    def __len__(self) -> int:\n        return len(self.paths)\n\n    def __getitem__(self, idx: int) -> dict:\n        image = self.transform(Image.open(self.paths[idx]).convert("RGB"))\n        return {"image": image, "is_video": False}\n\n\nclass MixedDataset(Dataset):\n    """\n    Interleaves video-frame pairs and static images.\n\n    Each __getitem__ samples from video_dataset with probability video_ratio\n    and from image_dataset otherwise.\n    """\n\n    def __init__(\n        self,\n        video_dataset: VideoFramePairDataset,\n        image_dataset: StaticImageDataset,\n        video_ratio: float = 0.5,\n        total_size: Optional[int] = None,\n    ) -> None:\n        super().__init__()\n        self.video_ds    = video_dataset\n        self.image_ds    = image_dataset\n        self.video_ratio = video_ratio\n        self._total      = total_size or max(len(video_dataset), len(image_dataset))\n\n    def __len__(self) -> int:\n        return self._total\n\n    def __getitem__(self, idx: int) -> dict:\n        if random.random() < self.video_ratio:\n            return self.video_ds[idx % len(self.video_ds)]\n        return self.image_ds[idx % len(self.image_ds)]\n\n\ndef mixed_collate_fn(batch: List[dict]) -> dict:\n    """Collate for homogeneous batches (all video or all image)."""\n    is_video = batch[0]["is_video"]\n    if is_video:\n        return {\n            "frame_i"  : torch.stack([b["frame_i"] for b in batch]),\n            "frame_j"  : torch.stack([b["frame_j"] for b in batch]),\n            "delta_t"  : torch.stack([b["delta_t"] for b in batch]),\n            "is_video" : True,\n        }\n    return {\n        "image"    : torch.stack([b["image"] for b in batch]),\n        "is_video" : False,\n    }\n\n\ndef build_dataloaders(\n    video_root: Optional[str] = None,\n    image_root: Optional[str] = None,\n    batch_size: int = 32,\n    num_workers: int = 4,\n    img_size: int = 224,\n    max_frame_gap: int = 48,\n    pin_memory: bool = True,\n) -> Tuple[Optional[DataLoader], Optional[DataLoader]]:\n    """Build separate DataLoaders for video and image datasets."""\n    video_loader = None\n    image_loader = None\n\n    if video_root is not None:\n        vds = VideoFramePairDataset(root=video_root, max_frame_gap=max_frame_gap, img_size=img_size)\n        video_loader = DataLoader(\n            vds, batch_size=batch_size, shuffle=True,\n            num_workers=num_workers, pin_memory=pin_memory,\n            drop_last=True, collate_fn=mixed_collate_fn,\n        )\n\n    if image_root is not None:\n        ids = StaticImageDataset(root=image_root, img_size=img_size)\n        image_loader = DataLoader(\n            ids, batch_size=batch_size, shuffle=True,\n            num_workers=num_workers, pin_memory=pin_memory,\n            drop_last=True, collate_fn=mixed_collate_fn,\n        )\n\n    return video_loader, image_loader\n',
    os.path.join(REPO_DIR, "engine_extended_pretrain.py"): '"""\nengine_extended_pretrain.py\n----------------------------\nTraining engine for the ViC-MAE temporal extension (Stage 1 & Stage 2).\n\nImplements the full mixed-loss training loop over alternating video and\nimage batches:\n\n  Video batch loss:\n      loss = L_mae(frame_i) + L_mae(frame_j)\n           + lambda_nce  * L_nce(pooled_i, pooled_j)\n           + lambda_cross * L_cross(ctx_i, frame_j, delta_t)\n\n  Image batch loss:\n      loss = L_mae(image)\n           + lambda_pres * L_pres(tokens, frozen_tokens)\n\nThe frozen_teacher is a deepcopy of the initial ViCMAE backbone kept in\neval mode with all gradients disabled.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport sys\nfrom copy import deepcopy\nfrom typing import Dict, Iterable, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom models.extended_vicmae import ExtendedViCMAE\n\n\n# ---------------------------------------------------------------------------\n# Loss weights (can be overridden via args)\n# ---------------------------------------------------------------------------\n\nDEFAULT_LAMBDA_NCE   = 0.1\nDEFAULT_LAMBDA_CROSS = 1.0\nDEFAULT_LAMBDA_PRES  = 0.5\n\n\n# ---------------------------------------------------------------------------\n# Training step\n# ---------------------------------------------------------------------------\n\ndef training_step(\n    model: ExtendedViCMAE,\n    frozen_teacher: nn.Module,\n    batch: Dict,\n    device: torch.device,\n    lambda_nce:   float = DEFAULT_LAMBDA_NCE,\n    lambda_cross: float = DEFAULT_LAMBDA_CROSS,\n    lambda_pres:  float = DEFAULT_LAMBDA_PRES,\n) -> Tuple[torch.Tensor, Dict[str, float]]:\n    """\n    Compute the combined loss for one batch.\n\n    Args:\n        model          : Active ExtendedViCMAE instance (training mode).\n        frozen_teacher : Frozen deepcopy of ViCMAE (eval, no grad).\n        batch          : Dict from mixed_collate_fn (video or image schema).\n        device         : Compute device.\n        lambda_nce     : Weight for InfoNCE contrastive loss.\n        lambda_cross   : Weight for cross-frame reconstruction loss.\n        lambda_pres    : Weight for feature preservation loss.\n\n    Returns:\n        loss       : Scalar combined loss.\n        loss_dict  : Dict of individual loss values for logging.\n    """\n    is_video = batch["is_video"]\n    loss_dict: Dict[str, float] = {}\n\n    if is_video:\n        frame_i = batch["frame_i"].to(device, non_blocking=True)\n        frame_j = batch["frame_j"].to(device, non_blocking=True)\n        delta_t = batch["delta_t"].to(device, non_blocking=True)\n\n        # --- L_mae for frame_i and frame_j (via base ViCMAE) ---\n        mae_i, _, _ = model.vicmae([frame_i, frame_i])   # symmetric pair\n        mae_j, _, _ = model.vicmae([frame_j, frame_j])\n        l_mae = 0.5 * (mae_i + mae_j)\n\n        # --- L_nce: encode both frames with base mask_ratio ---\n        mask_ratio = model.vicmae.mask_ratio\n        enc_i, _, _ = model.vicmae.forward_encoder(frame_i, mask_ratio)\n        enc_j, _, _ = model.vicmae.forward_encoder(frame_j, mask_ratio)\n\n        pool_i = model.vicmae.projector(enc_i[:, 1:, :].mean(dim=1))\n        pool_j = model.vicmae.projector(enc_j[:, 1:, :].mean(dim=1))\n        z      = torch.cat([pool_i, pool_j], dim=0)        # (2B, proj_dim)\n        l_nce  = model.vicmae.info_nce_loss(z, model.vicmae.temperature)\n\n        # --- L_cross: SiamMAE reconstruction of frame_j ---\n        # Use FULL (unmasked) Frame-i encoder tokens as context\n        ctx_i  = model.encode_full(frame_i)                # (B, N+1, D)\n        l_cross_res = model.forward_cross_recon(\n            ctx_tokens=ctx_i,\n            target_imgs=frame_j,\n            delta_t=delta_t,\n            mask_ratio=model.cross_mask_ratio,\n        )\n        l_cross = l_cross_res[0] if isinstance(l_cross_res, (tuple, list)) else l_cross_res\n\n        loss = l_mae + lambda_nce * l_nce + lambda_cross * l_cross\n        loss_dict = {\n            "loss_mae"  : l_mae.item(),\n            "loss_nce"  : l_nce.item(),\n            "loss_cross": l_cross.item(),\n        }\n\n    else:\n        image = batch["image"].to(device, non_blocking=True)\n\n        # --- L_mae for image (single view) ---\n        mae_loss, _, _ = model.vicmae([image, image])\n        l_mae = mae_loss\n\n        # --- L_pres: cosine distance to frozen teacher ---\n        with torch.no_grad():\n            frozen_tokens = frozen_teacher.forward_encoder(\n                image, mask_ratio=0.0\n            )[0].detach()\n\n        current_tokens = model.encode_full(image)           # (B, N+1, D)\n        l_pres = ExtendedViCMAE.compute_preservation_loss(current_tokens, frozen_tokens)\n        del frozen_tokens\n\n        # Cross-decoder reconstruction on image with delta_t = 0 (trains decoder in Stage 1)\n        delta_zero = torch.zeros(image.size(0), 1, device=device)\n        l_cross_res = model.forward_cross_recon(\n            ctx_tokens=current_tokens[:, 1:, :],\n            target_imgs=image,\n            delta_t=delta_zero,\n            mask_ratio=model.cross_mask_ratio,\n        )\n        l_cross = l_cross_res[0] if isinstance(l_cross_res, (tuple, list)) else l_cross_res\n\n        loss = l_mae + lambda_pres * l_pres + lambda_cross * l_cross\n        loss_dict = {\n            "loss_mae"  : l_mae.item(),\n            "loss_pres" : l_pres.item(),\n            "loss_cross": l_cross.item(),\n        }\n\n    loss_dict["loss_total"] = loss.item()\n    return loss, loss_dict\n\n\n# ---------------------------------------------------------------------------\n# One epoch\n# ---------------------------------------------------------------------------\n\ndef train_one_epoch_extended(\n    model: ExtendedViCMAE,\n    frozen_teacher: nn.Module,\n    video_loader: Optional[Iterable],\n    image_loader: Optional[Iterable],\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    epoch: int,\n    scaler: Optional[torch.cuda.amp.GradScaler] = None,\n    accum_iter: int = 1,\n    lambda_nce:   float = DEFAULT_LAMBDA_NCE,\n    lambda_cross: float = DEFAULT_LAMBDA_CROSS,\n    lambda_pres:  float = DEFAULT_LAMBDA_PRES,\n    log_freq: int = 20,\n) -> Dict[str, float]:\n    """\n    Train for one epoch over video and/or image DataLoaders.\n\n    Alternates between video and image batches within each step.\n    """\n    model.train()\n    frozen_teacher.eval()\n\n    # Build a unified iterator: interleave video and image batches\n    loaders = [l for l in (video_loader, image_loader) if l is not None]\n    if not loaders:\n        raise ValueError("At least one of video_loader or image_loader must be provided.")\n\n    batches     = [iter(l) for l in loaders]\n    total_steps = max(len(l) for l in loaders)\n\n    running: Dict[str, float] = {}\n    counts:  Dict[str, int]   = {}\n\n    optimizer.zero_grad()\n\n    for step in range(total_steps):\n        # Alternate between loaders; cycle shorter one\n        loader_idx = step % len(loaders)\n        try:\n            batch = next(batches[loader_idx])\n        except StopIteration:\n            batches[loader_idx] = iter(loaders[loader_idx])\n            batch = next(batches[loader_idx])\n\n        use_amp = scaler is not None\n        device_type = "cuda" if device.type == "cuda" else "cpu"\n        try:\n            amp_ctx = torch.amp.autocast(device_type=device_type, enabled=use_amp)\n        except (AttributeError, TypeError):\n            amp_ctx = torch.cuda.amp.autocast(enabled=use_amp)\n\n        with amp_ctx:\n            loss, loss_dict = training_step(\n                model, frozen_teacher, batch, device,\n                lambda_nce=lambda_nce,\n                lambda_cross=lambda_cross,\n                lambda_pres=lambda_pres,\n            )\n\n        if not math.isfinite(loss.item()):\n            print(f"[Epoch {epoch}] Step {step}: Loss is {loss.item()}, stopping.")\n            sys.exit(1)\n\n        loss = loss / accum_iter\n        if loss.requires_grad:\n            if use_amp:\n                scaler.scale(loss).backward()\n            else:\n                loss.backward()\n\n            if (step + 1) % accum_iter == 0 or (step + 1) == total_steps:\n                if use_amp:\n                    scaler.step(optimizer)\n                    scaler.update()\n                else:\n                    optimizer.step()\n                optimizer.zero_grad()\n\n        # Accumulate for logging\n        for k, v in loss_dict.items():\n            running[k] = running.get(k, 0.0) + v\n            counts[k]  = counts.get(k, 0) + 1\n\n        if step % log_freq == 0:\n            lr = optimizer.param_groups[0]["lr"]\n            parts = " | ".join(f"{k}: {v:.4f}" for k, v in loss_dict.items())\n            print(f"[Epoch {epoch}] Step {step}/{total_steps} | lr: {lr:.2e} | {parts}")\n\n    return {k: running[k] / max(counts[k], 1) for k in running}\n\n\n# ---------------------------------------------------------------------------\n# Optimizer builders\n# ---------------------------------------------------------------------------\n\ndef build_stage1_optimizer(\n    model: ExtendedViCMAE,\n    lr: float = 1e-4,\n    weight_decay: float = 0.05,\n) -> torch.optim.AdamW:\n    """\n    Stage 1: only the cross_decoder trains.\n    Backbone must already be frozen via model.freeze_backbone().\n    """\n    params = [p for p in model.cross_decoder.parameters() if p.requires_grad]\n    assert params, "No trainable parameters in cross_decoder. Did you call freeze_backbone()?"\n    return torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=weight_decay)\n\n\ndef build_stage2_optimizer(\n    model: ExtendedViCMAE,\n    backbone_lr: float = 1e-5,\n    decoder_lr:  float = 5e-5,\n    weight_decay: float = 0.05,\n) -> torch.optim.AdamW:\n    """\n    Stage 2: backbone (unfrozen portions) and cross_decoder train at different lrs.\n    """\n    decoder_ids = set(id(p) for p in model.cross_decoder.parameters())\n    backbone_params = [\n        p for p in model.vicmae.parameters()\n        if p.requires_grad and id(p) not in decoder_ids\n    ]\n    decoder_params = [p for p in model.cross_decoder.parameters() if p.requires_grad]\n\n    param_groups = []\n    if backbone_params:\n        param_groups.append({"params": backbone_params, "lr": backbone_lr})\n    if decoder_params:\n        param_groups.append({"params": decoder_params, "lr": decoder_lr})\n\n    assert param_groups, "No trainable parameters found for Stage 2 optimizer."\n    return torch.optim.AdamW(param_groups, betas=(0.9, 0.95), weight_decay=weight_decay)\n',
    os.path.join(REPO_DIR, "train_extended.py"): '"""\ntrain_extended.py\n-----------------\nTop-level training script for the ViC-MAE temporal extension.\n\nTraining protocol:\n  Main Training (30 epochs by default, backbone_lr=1e-5, decoder_lr=5e-5):\n    - Encoder adapted (LoRA adapters or top-4 blocks) + CrossFrameDecoder.\n    - Full 4-loss training with feature preservation.\n\n  Optional Stage 1 Warmup (stage1_epochs > 0, default: 0):\n    - Backbone frozen.\n    - Only CrossFrameDecoder trained.\n\nUsage:\n    python train_extended.py \\\\\n        --checkpoint path/to/vicmae_pretrained.pth \\\\\n        --video_root  path/to/video_frames \\\\\n        --image_root  path/to/tiny_imagenet/train \\\\\n        --stage1_epochs 5 \\\\\n        --stage2_epochs 20 \\\\\n        --use_lora \\\\\n        --output_dir output/extended\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport datetime\nimport json\nimport os\nimport sys\nimport time\nfrom copy import deepcopy\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\nimport torch.backends.cudnn as cudnn\n\n# Allow running from ViC-MAE sub-directory\n_HERE = os.path.dirname(os.path.abspath(__file__))\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\n\nimport models_vicmae\nfrom models.extended_vicmae import ExtendedViCMAE\nfrom engine_extended_pretrain import (\n    train_one_epoch_extended,\n    build_stage1_optimizer,\n    build_stage2_optimizer,\n)\nfrom datasets.video_image_dataset import build_dataloaders\n\n\n# ---------------------------------------------------------------------------\n# Argument parser\n# ---------------------------------------------------------------------------\n\ndef get_args() -> argparse.Namespace:\n    p = argparse.ArgumentParser("ViC-MAE Temporal Extension Training", add_help=True)\n\n    # Checkpoint\n    p.add_argument("--checkpoint", type=str, default="",\n                   help="Path to pretrained ViC-MAE checkpoint (.pth).")\n    p.add_argument("--vicmae_arch", type=str, default="vicmae_vit_base_patch16",\n                   help="ViCMAE architecture name (must match checkpoint).")\n\n    # Data\n    p.add_argument("--video_root", type=str, default=None,\n                   help="Root directory of extracted video frames.")\n    p.add_argument("--image_root", type=str, default=None,\n                   help="Root directory of static images (e.g., TinyImageNet).")\n    p.add_argument("--img_size", type=int, default=224)\n    p.add_argument("--max_frame_gap", type=int, default=48,\n                   help="Maximum frame-index gap for delta_t normalisation.")\n\n    # Training\n    p.add_argument("--epochs",       type=int, default=30,\n                   help="Total training epochs across Stage 1 + Stage 2 (default: 30).")\n    p.add_argument("--batch_size",   type=int, default=16)\n    p.add_argument("--num_workers",  type=int, default=4)\n    p.add_argument("--stage1_epochs",type=int, default=0,\n                   help="Epochs for Stage 1 (decoder warm-up, backbone frozen; default: 0).")\n    p.add_argument("--stage2_epochs",type=int, default=30,\n                   help="Epochs for Stage 2 / main training (joint fine-tuning; default: 30).")\n    p.add_argument("--stage1_lr",    type=float, default=1e-4)\n    p.add_argument("--backbone_lr",  type=float, default=1e-5)\n    p.add_argument("--decoder_lr",   type=float, default=5e-5)\n    p.add_argument("--weight_decay", type=float, default=0.05)\n    p.add_argument("--accum_iter",   type=int,   default=4)\n    p.add_argument("--use_amp",      action="store_true", default=False,\n                   help="Enable AMP (automatic mixed precision).")\n\n    # Stage 2 mode\n    p.add_argument("--use_lora",     action="store_true", default=False,\n                   help="Use LoRA adapters for Stage 2 (default: top-4 blocks).")\n    p.add_argument("--lora_r",       type=int,   default=16)\n    p.add_argument("--lora_alpha",   type=float, default=32.0)\n    p.add_argument("--top_blocks",   type=int,   default=4,\n                   help="Number of top ViT blocks to unfreeze in Stage 2.")\n\n    # Loss weights\n    p.add_argument("--lambda_nce",   type=float, default=0.1)\n    p.add_argument("--lambda_cross", type=float, default=1.0)\n    p.add_argument("--lambda_pres",  type=float, default=0.5)\n    # Convenience aliases\n    p.add_argument("--lr",                type=float, default=None, help="Global lr alias.")\n    p.add_argument("--min_lr",            type=float, default=1e-6)\n    p.add_argument("--warmup_epochs",     type=int,   default=2)\n    p.add_argument("--log_freq",          type=int,   default=20)\n    p.add_argument("--loss_weight_mae",   type=float, default=1.0)\n    p.add_argument("--loss_weight_cross", type=float, default=None)\n    p.add_argument("--loss_weight_pres",  type=float, default=None)\n\n    # CrossDecoder config\n    p.add_argument("--decoder_dim",   type=int, default=512)\n    p.add_argument("--decoder_depth", type=int, default=4)\n    p.add_argument("--decoder_heads", type=int, default=8)\n\n    # Output\n    p.add_argument("--output_dir", type=str, default="output/extended")\n    p.add_argument("--seed",       type=int, default=42)\n    p.add_argument("--device",     type=str, default="cuda")\n    p.add_argument("--resume",     type=str, default="",\n                   help="Path to resume checkpoint (.pth).")\n    p.add_argument("--resume_expected_epoch", type=int, default=None,\n                   help="Require this saved zero-based epoch index before resuming.")\n    p.add_argument("--start_epoch", type=int, default=None,\n                   help="Explicit start epoch index (e.g. 10 to continue after 0-9).")\n    p.add_argument("--save_freq",   type=int, default=1,\n                   help="Save checkpoint every N epochs (default: 1).")\n\n    args = p.parse_args()\n    if args.lr is not None:\n        args.stage1_lr = args.lr\n        args.decoder_lr = args.lr\n    if args.loss_weight_cross is not None:\n        args.lambda_cross = args.loss_weight_cross\n    if args.loss_weight_pres is not None:\n        args.lambda_pres = args.loss_weight_pres\n    return args\n\n\n# ---------------------------------------------------------------------------\n# Checkpoint helpers\n# ---------------------------------------------------------------------------\n\ndef save_checkpoint(\n    model: ExtendedViCMAE,\n    optimizer: torch.optim.Optimizer,\n    epoch: int,\n    output_dir: str,\n    tag: str = "",\n    scaler=None,\n) -> None:\n    Path(output_dir).mkdir(parents=True, exist_ok=True)\n    fname = f"checkpoint_epoch{epoch:04d}{tag}.pth"\n    state = {\n        "epoch"     : epoch,\n        "model"     : model.state_dict(),\n        "optimizer" : optimizer.state_dict(),\n    }\n    if scaler is not None:\n        state["scaler"] = scaler.state_dict()\n    # Publish completed file via temporary file to prevent corruption\n    destination = os.path.join(output_dir, fname)\n    temporary = destination + ".tmp"\n    torch.save(state, temporary)\n    os.replace(temporary, destination)\n\n    # Also update checkpoint_latest.pth\n    latest_path = os.path.join(output_dir, "checkpoint_latest.pth")\n    temp_latest = latest_path + ".tmp"\n    torch.save(state, temp_latest)\n    os.replace(temp_latest, latest_path)\n    print(f"  Saved checkpoint: {fname} (and updated checkpoint_latest.pth)")\n\n\ndef restore_training_checkpoint(\n    model: ExtendedViCMAE,\n    optimizer: torch.optim.Optimizer,\n    scaler,\n    checkpoint_path: str,\n    expected_epoch: Optional[int] = None,\n    end_epoch: int = 30,\n    start_epoch_override: Optional[int] = None,\n    device: torch.device = torch.device("cpu"),\n) -> int:\n    """Restore complete training state, or stop with clear error."""\n    if not Path(checkpoint_path).is_file():\n        raise FileNotFoundError(\n            f"Resume checkpoint not found: {checkpoint_path}. "\n            "Please ensure the checkpoint dataset is attached and path is correct."\n        )\n    try:\n        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    except Exception:\n        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n\n    if not isinstance(ckpt, dict):\n        raise ValueError("Resume checkpoint must contain a training-state dictionary.")\n    missing = {"model", "optimizer", "epoch"} - ckpt.keys()\n    if missing:\n        raise ValueError(f"Incomplete resume checkpoint; missing keys: {sorted(missing)}")\n    saved_epoch = ckpt["epoch"]\n    if isinstance(saved_epoch, bool) or not isinstance(saved_epoch, int):\n        raise ValueError(f"Saved epoch must be an integer index, got {saved_epoch!r}")\n    if expected_epoch is not None and saved_epoch != expected_epoch:\n        raise ValueError(f"Expected saved epoch index {expected_epoch}, got {saved_epoch}")\n\n    if start_epoch_override is not None:\n        start_epoch = start_epoch_override\n    else:\n        start_epoch = saved_epoch + 1\n\n    if not 0 <= start_epoch < end_epoch:\n        raise ValueError(f"No valid remaining epochs: start={start_epoch}, stop={end_epoch}")\n    if not isinstance(ckpt["model"], dict) or not ckpt["model"]:\n        raise ValueError("Resume checkpoint has no model weights.")\n    optimizer_state = ckpt["optimizer"]\n    if (not isinstance(optimizer_state, dict)\n            or not optimizer_state.get("state")\n            or not optimizer_state.get("param_groups")):\n        raise ValueError("Resume checkpoint has no trained optimizer state.")\n\n    # Strict loading ensures 100% parameter alignment\n    model.load_state_dict(ckpt["model"], strict=True)\n    optimizer.load_state_dict(optimizer_state)\n    for state in optimizer.state.values():\n        for k, v in state.items():\n            if isinstance(v, torch.Tensor):\n                state[k] = v.to(device)\n\n    if scaler is not None:\n        if ckpt.get("scaler"):\n            scaler.load_state_dict(ckpt["scaler"])\n            print("  Restored AMP scaler state.")\n        else:\n            print("  NOTE: Checkpoint has no AMP scaler state; using fresh GradScaler.")\n\n    print(f"  Restored complete model, optimizer, and scaler from {checkpoint_path}")\n    print(f"  Saved epoch index: {saved_epoch} (completed {saved_epoch + 1} epochs); resuming indices "\n          f"{start_epoch}..{end_epoch - 1} ({end_epoch - start_epoch} epochs to train).")\n    print(f"  First displayed epoch: {start_epoch + 1}/{end_epoch}; "\n          f"optimizer LRs: {[group[\'lr\'] for group in optimizer.param_groups]}")\n    return start_epoch\n\n\ndef load_vicmae_weights(model: ExtendedViCMAE, ckpt_path: str) -> None:\n    """Load pretrained ViCMAE weights into the model\'s vicmae sub-module."""\n    try:\n        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)\n    except Exception:\n        ckpt = torch.load(ckpt_path, map_location="cpu")\n    # Support common checkpoint formats\n    state = ckpt.get("model", ckpt.get("state_dict", ckpt))\n    # Strip \'module.\' prefix from DDP checkpoints\n    state = {k.replace("module.", ""): v for k, v in state.items()}\n    missing, unexpected = model.vicmae.load_state_dict(state, strict=False)\n    print(f"  Loaded ViCMAE weights from {ckpt_path}")\n    if missing:\n        print(f"  Missing keys ({len(missing)}): {missing[:5]} ...")\n    if unexpected:\n        print(f"  Unexpected keys ({len(unexpected)}): {unexpected[:5]} ...")\n\n\n# ---------------------------------------------------------------------------\n# Main\n# ---------------------------------------------------------------------------\n\ndef main() -> None:\n    args = get_args()\n    device = torch.device(args.device if torch.cuda.is_available() else "cpu")\n\n    torch.manual_seed(args.seed)\n    cudnn.benchmark = True\n\n    # -----------------------------------------------------------------------\n    # Build model\n    # -----------------------------------------------------------------------\n    print("\\n==> Building ExtendedViCMAE ...")\n    # Resolve ViCMAE constructor kwargs from architecture name\n    arch_defaults = {\n        "vicmae_vit_base_patch16" : dict(embed_dim=768,  depth=12, num_heads=12,\n                                         decoder_embed_dim=512, decoder_depth=8,\n                                         decoder_num_heads=16),\n        "vicmae_vit_large_patch16": dict(embed_dim=1024, depth=24, num_heads=16,\n                                         decoder_embed_dim=512, decoder_depth=8,\n                                         decoder_num_heads=16),\n        "vicmae_vit_small_patch16": dict(embed_dim=384,  depth=12, num_heads=6,\n                                         decoder_embed_dim=256, decoder_depth=8,\n                                         decoder_num_heads=8),\n    }\n    vicmae_cfg = arch_defaults.get(args.vicmae_arch, {})\n\n    model = ExtendedViCMAE(\n        vicmae_cfg=vicmae_cfg,\n        decoder_dim=args.decoder_dim,\n        decoder_depth=args.decoder_depth,\n        decoder_heads=args.decoder_heads,\n        img_size=args.img_size,\n        cross_mask_ratio=0.95,\n    ).to(device)\n\n    # Load pretrained ViCMAE backbone\n    if args.checkpoint:\n        load_vicmae_weights(model, args.checkpoint)\n\n    # Frozen teacher: deepcopy of the pretrained ViCMAE, frozen forever\n    frozen_teacher = deepcopy(model.vicmae).to(device)\n    frozen_teacher.eval()\n    frozen_teacher.requires_grad_(False)\n\n    # -----------------------------------------------------------------------\n    # Build data loaders\n    # -----------------------------------------------------------------------\n    print("\\n==> Building data loaders ...")\n    video_loader, image_loader = build_dataloaders(\n        video_root=args.video_root,\n        image_root=args.image_root,\n        batch_size=args.batch_size,\n        num_workers=args.num_workers,\n        img_size=args.img_size,\n        max_frame_gap=args.max_frame_gap,\n    )\n\n    if args.use_amp:\n        try:\n            scaler = torch.amp.GradScaler("cuda")\n        except (AttributeError, TypeError):\n            scaler = torch.cuda.amp.GradScaler()\n    else:\n        scaler = None\n\n    total_epochs = args.stage1_epochs + args.stage2_epochs\n\n    # -----------------------------------------------------------------------\n    # Stage 1: Decoder warm-up (backbone frozen)\n    # -----------------------------------------------------------------------\n    if args.stage1_epochs > 0:\n        print(f"\\n{\'=\'*60}")\n        print(f"  STAGE 1: Decoder warm-up for {args.stage1_epochs} epochs (Epochs 1-{args.stage1_epochs} of {total_epochs})")\n        print(f"{\'=\'*60}")\n\n        model.freeze_backbone()\n        optimizer = build_stage1_optimizer(\n            model, lr=args.stage1_lr, weight_decay=args.weight_decay\n        )\n\n        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n        print(f"  Trainable params: {trainable:,}")\n\n        log_history = []\n        for epoch in range(args.stage1_epochs):\n            t0 = time.time()\n            metrics = train_one_epoch_extended(\n                model=model,\n                frozen_teacher=frozen_teacher,\n                video_loader=video_loader,\n                image_loader=image_loader,\n                optimizer=optimizer,\n                device=device,\n                epoch=epoch,\n                scaler=scaler,\n                accum_iter=args.accum_iter,\n                lambda_nce=args.lambda_nce,\n                lambda_cross=args.lambda_cross,\n                lambda_pres=args.lambda_pres,\n            )\n            elapsed = time.time() - t0\n            global_epoch = epoch + 1\n            log = {"stage": 1, "epoch": global_epoch, "stage_epoch": epoch, "time_s": elapsed, **metrics}\n            log_history.append(log)\n            with open(os.path.join(args.output_dir, "train_extended_log.json"), "a") as f_log:\n                f_log.write(json.dumps(log) + "\\n")\n            print(f"  [Epoch {global_epoch}/{total_epochs} (Stage 1)] {elapsed:.1f}s | " +\n                  " | ".join(f"{k}: {v:.4f}" for k, v in metrics.items()))\n\n            if (epoch + 1) % args.save_freq == 0 or epoch + 1 == args.stage1_epochs:\n                save_checkpoint(model, optimizer, epoch, args.output_dir, tag="_s1", scaler=scaler)\n\n        # Save Stage 1 log\n        with open(os.path.join(args.output_dir, "log_stage1.json"), "w") as f:\n            json.dump(log_history, f, indent=2)\n\n    # -----------------------------------------------------------------------\n    # Stage 2: Joint fine-tuning\n    # -----------------------------------------------------------------------\n    if args.stage2_epochs > 0:\n        if args.stage1_epochs > 0:\n            s2_start_disp = args.stage1_epochs + 1\n            s2_end_disp = total_epochs\n            print(f"\\n{\'=\'*60}")\n            print(f"  STAGE 2: Joint fine-tuning for {args.stage2_epochs} epochs (Epochs {s2_start_disp}-{s2_end_disp} of {total_epochs})")\n            print(f"{\'=\'*60}")\n        else:\n            print(f"\\n{\'=\'*60}")\n            print(f"  MAIN TRAINING: Joint fine-tuning for {args.stage2_epochs} epochs (Epochs 1-{total_epochs})")\n            print(f"{\'=\'*60}")\n\n        # Unfreeze backbone portion\n        if args.use_lora:\n            print(f"  Strategy: LoRA (r={args.lora_r}, alpha={args.lora_alpha})")\n            model.inject_lora_adapters(r=args.lora_r, alpha=args.lora_alpha)\n        else:\n            print(f"  Strategy: Unfreeze top {args.top_blocks} ViT blocks")\n            model.unfreeze_top_blocks(n=args.top_blocks)\n\n        # Cross decoder should always be trainable in Stage 2\n        model.cross_decoder.requires_grad_(True)\n        model.to(device)\n\n        optimizer = build_stage2_optimizer(\n            model,\n            backbone_lr=args.backbone_lr,\n            decoder_lr=args.decoder_lr,\n            weight_decay=args.weight_decay,\n        )\n\n        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n        print(f"  Trainable params: {trainable:,}")\n\n        # Resume if requested\n        start_epoch = 0\n        if args.resume:\n            start_epoch = restore_training_checkpoint(\n                model=model,\n                optimizer=optimizer,\n                scaler=scaler,\n                checkpoint_path=args.resume,\n                expected_epoch=args.resume_expected_epoch,\n                end_epoch=args.stage2_epochs,\n                start_epoch_override=args.start_epoch,\n                device=device,\n            )\n\n        log_history = []\n        for epoch in range(start_epoch, args.stage2_epochs):\n            t0 = time.time()\n            metrics = train_one_epoch_extended(\n                model=model,\n                frozen_teacher=frozen_teacher,\n                video_loader=video_loader,\n                image_loader=image_loader,\n                optimizer=optimizer,\n                device=device,\n                epoch=epoch,\n                scaler=scaler,\n                accum_iter=args.accum_iter,\n                lambda_nce=args.lambda_nce,\n                lambda_cross=args.lambda_cross,\n                lambda_pres=args.lambda_pres,\n            )\n            elapsed = time.time() - t0\n            global_epoch = args.stage1_epochs + epoch + 1\n            log = {"stage": 2, "epoch": global_epoch, "stage_epoch": epoch, "time_s": elapsed, **metrics}\n            log_history.append(log)\n            with open(os.path.join(args.output_dir, "train_extended_log.json"), "a") as f_log:\n                f_log.write(json.dumps(log) + "\\n")\n            stage_label = "Stage 2" if args.stage1_epochs > 0 else "Main"\n            print(f"  [Epoch {global_epoch}/{total_epochs} ({stage_label})] {elapsed:.1f}s | " +\n                  " | ".join(f"{k}: {v:.4f}" for k, v in metrics.items()))\n\n            if (epoch + 1) % args.save_freq == 0 or (epoch + 1) == args.stage2_epochs:\n                save_checkpoint(model, optimizer, epoch, args.output_dir, tag="_s2", scaler=scaler)\n\n        with open(os.path.join(args.output_dir, "log_stage2.json"), "w") as f:\n            json.dump(log_history, f, indent=2)\n\n    print("\\nTraining complete.")\n\n\nif __name__ == "__main__":\n    main()\n',
    os.path.join(REPO_DIR, "test_extension.py"): '"""Verification & smoke tests for the extended ViC-MAE.\n\nRuns dummy forward/backward passes on random synthetic tensors and checks:\n\n* Cross-frame decoder output shapes and the ~95% binary mask.\n* Video-batch loss composition (L_mae + L_nce + L_cross) and backward.\n* Image-batch preservation loss (L_mae + L_pres) and backward.\n* Stage 1 gradient isolation: only ``cross_decoder`` receives gradients.\n* Stage 2 gradient propagation: both ``cross_decoder`` and the backbone\n  (LoRA) receive gradients.\n* No NaNs or dimension mismatches in the cross-attention queries.\n\nUsage:\n    python test_extension.py            # standalone runner\n    pytest test_extension.py -v        # pytest\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom types import SimpleNamespace\n\nimport torch\nimport torch.nn as nn\n\nfrom models import (\n    CrossFrameDecoder,\n    ExtendedViCMAE,\n    inject_lora,\n    mark_only_lora_trainable,\n)\nfrom models_vicmae import ViCMAE\n\nB = 2\nIMG = 224\nEMBED_DIM = 384\nDECODER_DIM = 256\nPATCH_SIZE = 16\nNUM_PATCHES = (IMG // PATCH_SIZE) ** 2\nPATCH_DIM = PATCH_SIZE * PATCH_SIZE * 3\n\n\ndef _tiny_base() -> ViCMAE:\n    """A small ViCMAE (random weights) so the smoke test runs fast on CPU."""\n    return ViCMAE(\n        embed_dim=EMBED_DIM,\n        depth=4,\n        num_heads=6,\n        decoder_embed_dim=DECODER_DIM,\n        decoder_depth=2,\n        decoder_num_heads=8,\n        patch_size=PATCH_SIZE,\n        norm_pix_loss=True,\n    )\n\n\ndef _video_batch(b: int = B, device: str = "cpu") -> dict:\n    return {\n        "frame_i": torch.randn(b, 3, IMG, IMG, device=device),\n        "frame_j": torch.randn(b, 3, IMG, IMG, device=device),\n        "delta_t": torch.rand(b, 1, device=device),\n        "is_video": True,\n        "mask_ratio": 0.5,\n        "lambda_nce": 0.1,\n        "lambda_cross": 1.0,\n    }\n\n\ndef _image_batch(b: int = B, device: str = "cpu") -> dict:\n    img = torch.randn(b, 3, IMG, IMG, device=device)\n    return {\n        "image": img,\n        "is_video": False,\n        "mask_ratio": 0.5,\n        "lambda_pres": 0.5,\n        "frozen_tokens": torch.randn(b, NUM_PATCHES, EMBED_DIM, device=device),\n    }\n\n\ndef _trainable_grads(module: nn.Module):\n    return [(n, p) for n, p in module.named_parameters()\n            if p.requires_grad and p.grad is not None]\n\n\ndef _any_nonzero_grad(module: nn.Module) -> bool:\n    return any(\n        p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0\n        for p in module.parameters()\n    )\n\n\n# ----------------------------------------------------------------------\n# Tests\n# ----------------------------------------------------------------------\ndef test_cross_decoder_shapes_and_mask():\n    dec = CrossFrameDecoder(embed_dim=EMBED_DIM, decoder_dim=DECODER_DIM,\n                            depth=4, num_heads=8, patch_size=PATCH_SIZE)\n    ctx = torch.randn(B, 10, EMBED_DIM)  # e.g. 10 visible tokens of frame_i\n    delta = torch.rand(B, 1)\n\n    pred = dec(ctx, delta)\n    assert pred.shape == (B, NUM_PATCHES, PATCH_DIM), f"pred shape {pred.shape}"\n\n    # Query construction shape check: mask_token + pos_embed + time_embed.\n    assert dec.mask_token.shape == (1, 1, DECODER_DIM)\n    assert dec.pos_embed.shape == (1, NUM_PATCHES, DECODER_DIM)\n    assert dec.time_embed[0].in_features == 1 and dec.time_embed[0].out_features == DECODER_DIM\n    assert torch.isfinite(pred).all(), "NaN/Inf in decoder output"\n    print("PASS test_cross_decoder_shapes_and_mask")\n\n\ndef test_cross_recon_mask_ratio_and_loss():\n    model = ExtendedViCMAE(_tiny_base())\n    model.train()\n\n    ctx = torch.randn(B, 10, EMBED_DIM)\n    target = torch.randn(B, 3, IMG, IMG)\n    delta = torch.rand(B, 1)\n\n    loss, pred, mask = model.forward_cross_recon(ctx, target, delta, mask_ratio=0.95)\n    assert loss.ndim == 0 and torch.isfinite(loss), f"loss {loss}"\n    assert pred.shape == (B, NUM_PATCHES, PATCH_DIM)\n    assert mask.shape == (B, NUM_PATCHES)\n    masked_frac = mask.mean().item()\n    assert 0.90 < masked_frac < 1.0, f"masked fraction {masked_frac}"\n    # Loss is zero only if nothing masked; with 95% masked it must be > 0.\n    assert loss.item() > 0\n    print("PASS test_cross_recon_mask_ratio_and_loss")\n\n\ndef test_video_batch_losses_and_backward():\n    model = ExtendedViCMAE(_tiny_base())\n    model.train()\n    batch = _video_batch()\n\n    out = model(batch)\n    expected_keys = {"loss", "loss_mae_i", "loss_mae_j", "loss_nce", "loss_cross"}\n    assert expected_keys.issubset(out.keys()), f"missing keys: {expected_keys - set(out)}"\n    assert all(torch.isfinite(v) for v in out.values()), "NaN/Inf in video losses"\n    assert out["loss"].item() > 0\n\n    out["loss"].backward()\n    assert _any_nonzero_grad(model.cross_decoder), "cross_decoder got no gradient"\n    print("PASS test_video_batch_losses_and_backward")\n\n\ndef test_image_batch_preservation_loss():\n    model = ExtendedViCMAE(_tiny_base())\n    model.train()\n    batch = _image_batch()\n\n    out = model(batch)\n    assert {"loss", "loss_mae", "loss_pres"}.issubset(out.keys())\n    assert 0.0 <= out["loss_pres"].item() <= 2.0, f"pres loss {out[\'loss_pres\']}"\n    assert torch.isfinite(out["loss"])\n\n    out["loss"].backward()\n    assert _any_nonzero_grad(model.cross_decoder) or _any_nonzero_grad(model.vicmae)\n    print("PASS test_image_batch_preservation_loss")\n\n\ndef test_preservation_loss_math():\n    model = ExtendedViCMAE(_tiny_base())\n    tok = torch.randn(B, NUM_PATCHES, EMBED_DIM)\n    # Identical tokens -> cosine similarity 1 -> loss 0.\n    assert model.compute_preservation_loss(tok, tok).item() == 0.0\n    # Opposed tokens -> cosine similarity -1 -> loss 2.\n    assert abs(model.compute_preservation_loss(tok, -tok).item() - 2.0) < 1e-5\n    print("PASS test_preservation_loss_math")\n\n\ndef test_stage1_gradient_isolation():\n    """Stage 1: frozen encoder -> only cross_decoder receives gradients."""\n    model = ExtendedViCMAE(_tiny_base())\n    for p in model.parameters():\n        p.requires_grad_(False)\n    for p in model.cross_decoder.parameters():\n        p.requires_grad_(True)\n\n    batch = _video_batch()\n    tok_i, _, _ = model.encode_frame(batch["frame_i"], 0.5)\n    loss, _, _ = model.forward_cross_recon(\n        tok_i[:, 1:, :], batch["frame_j"], batch["delta_t"], mask_ratio=0.95\n    )\n    loss.backward()\n\n    assert _any_nonzero_grad(model.cross_decoder), "cross_decoder got no gradient"\n    vicmae_grads = _trainable_grads(model.vicmae)\n    assert len(vicmae_grads) == 0, f"frozen encoder got gradients: {vicmae_grads[:3]}"\n    print("PASS test_stage1_gradient_isolation")\n\n\ndef test_stage2_lora_gradient_propagation():\n    """Stage 2 (LoRA): both cross_decoder and backbone LoRA get gradients."""\n    model = ExtendedViCMAE(_tiny_base())\n    inject_lora(model.vicmae.blocks, r=16, alpha=32)\n    mark_only_lora_trainable(model.vicmae.blocks)\n    for p in model.cross_decoder.parameters():\n        p.requires_grad_(True)\n\n    batch = _video_batch()\n    out = model(batch)\n    out["loss"].backward()\n\n    assert _any_nonzero_grad(model.cross_decoder), "cross_decoder got no gradient"\n    lora_grads = [(n, p) for n, p in model.vicmae.named_parameters()\n                  if "lora_" in n and p.grad is not None]\n    assert len(lora_grads) > 0, "LoRA params got no gradient"\n    assert all(torch.isfinite(p.grad).all() for _, p in lora_grads), "NaN in LoRA grads"\n    print("PASS test_stage2_lora_gradient_propagation")\n\n\ndef test_cross_attention_query_no_nan():\n    """Explicitly exercise the query path and assert finite, correctly shaped queries."""\n    dec = CrossFrameDecoder(embed_dim=EMBED_DIM, decoder_dim=DECODER_DIM,\n                            depth=4, num_heads=8, patch_size=PATCH_SIZE)\n    ctx = torch.randn(B, 10, EMBED_DIM)\n    delta = torch.rand(B, 1)\n\n    # Reconstruct the exact query tensor the decoder builds.\n    time_emb = dec.time_embed(delta.float())\n    query = dec.mask_token + dec.pos_embed + time_emb.unsqueeze(1)\n    query = query.expand(B, NUM_PATCHES, DECODER_DIM)\n    assert query.shape == (B, NUM_PATCHES, DECODER_DIM), f"query shape {query.shape}"\n    assert torch.isfinite(query).all(), "NaN in cross-attention queries"\n\n    # Run a full block and check the attention output shape.\n    from models import CrossAttentionBlock\n    block = CrossAttentionBlock(DECODER_DIM, 8)\n    context = dec.proj(ctx)\n    out = block(query, context)\n    assert out.shape == (B, NUM_PATCHES, DECODER_DIM)\n    assert torch.isfinite(out).all(), "NaN after cross-attention block"\n    print("PASS test_cross_attention_query_no_nan")\n\n\ndef test_video_dataset_raw_video_loading(tmp_path=None):\n    """Verify that VideoFramePairDataset can load directly from a raw video file using cv2."""\n    import tempfile\n    import numpy as np\n    import cv2\n    from pathlib import Path\n    from datasets.video_image_dataset import VideoFramePairDataset\n\n    tmp_dir = Path(tempfile.mkdtemp())\n    video_file = tmp_dir / "test_clip.mp4"\n\n    # Create a small dummy MP4 video with 10 synthetic frames\n    fourcc = cv2.VideoWriter_fourcc(*"mp4v")\n    out = cv2.VideoWriter(str(video_file), fourcc, 10.0, (64, 64))\n    for i in range(10):\n        frame = np.full((64, 64, 3), fill_value=i * 25, dtype=np.uint8)\n        out.write(frame)\n    out.release()\n\n    # Load dataset pointing directly to the video file\n    ds = VideoFramePairDataset(root=str(tmp_dir), img_size=224, max_frame_gap=5)\n    assert len(ds) == 1\n    sample = ds[0]\n\n    assert sample["is_video"] is True\n    assert sample["frame_i"].shape == (3, 224, 224)\n    assert sample["frame_j"].shape == (3, 224, 224)\n    assert sample["delta_t"].shape == (1,)\n    assert 0.0 <= sample["delta_t"].item() <= 1.0\n    print("PASS test_video_dataset_raw_video_loading")\n\n\n# ----------------------------------------------------------------------\n# Standalone runner\n# ----------------------------------------------------------------------\ndef _run_all():\n    tests = [\n        test_cross_decoder_shapes_and_mask,\n        test_cross_recon_mask_ratio_and_loss,\n        test_video_batch_losses_and_backward,\n        test_image_batch_preservation_loss,\n        test_preservation_loss_math,\n        test_stage1_gradient_isolation,\n        test_stage2_lora_gradient_propagation,\n        test_cross_attention_query_no_nan,\n        test_video_dataset_raw_video_loading,\n    ]\n    for t in tests:\n        t()\n    print("\\n[ALL SMOKE TESTS PASSED]")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--test", default=None, help="run a single test by name")\n    args = parser.parse_args()\n    if args.test:\n        globals()[args.test]()\n        print(f"\\n[PASSED] {args.test}")\n    else:\n        _run_all()\n',
}

for fpath, code in files.items():
    with open(fpath, "w", encoding="utf-8") as f:
        f.write(code)
    print(f"Written: {fpath} ({len(code)} bytes)")

print("\n✅ All extension modules deployed successfully!")

## 4. Run Verification Smoke Tests
Runs all 12 unit tests to verify cross-attention decoder, LoRA, preservation loss, and training steps on the Kaggle GPU environment.

In [ ]:
!cd /kaggle/working/ViC-MAE && pytest test_extension.py -v

## 5. Launch Extended Pretraining (Resume from Epoch 10 to 30)
- **Resume from Epoch 10:** Loads checkpoint `checkpoint_epoch0009_s2.pth` (saved after running epochs 0 to 9) and resumes Stage 2 joint fine-tuning directly from epoch 10 up to epoch 30.
- **Auto Checkpoint Resolution:** Searches the specified path (`/kaggle/input/datasets/tirthmanavadariya/vicmae-checkpoint/checkpoint_epoch0009_s2.pth`), standard Kaggle dataset mount (`/kaggle/input/vicmae-checkpoint/checkpoint_epoch0009_s2.pth`), and recursive glob search in `/kaggle/input/`.
- **Log History Preservation:** If `train_extended_log.json` exists in the checkpoint directory, it is automatically restored to preserve full curve history from epoch 1 to 30.
- **Joint Optimization:** Trains Cross-Attention Decoder + LoRA ($r=16$) backbone adapters alongside Feature Preservation Loss ($L_{pres}$) against the frozen ViC-MAE checkpoint.
- **Mixed Precision (`--use_amp`):** Enabled for fast GPU computation.

In [ ]:
import os, sys, torch, glob, shutil
from pathlib import Path

REPO_DIR = "/kaggle/working/ViC-MAE"
CKPT_PATH = "/kaggle/working/vicmae_pretrain_vit_base_in1k_k400.pth"
IMAGE_DIR = "/kaggle/working/tiny-imagenet-200/train"
OUTPUT_DIR = "/kaggle/working/output/vicmae_extended"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# -----------------------------------------------------------------
# 1. Resume Checkpoint Setup (Continuing from Epoch 10)
# -----------------------------------------------------------------
START_EPOCH = 10     # Resuming from epoch 10 (epochs 0-9 already completed)
TOTAL_EPOCHS = 30    # Target total epochs (epochs 10 to 29 will train)

user_ckpt_path = "/kaggle/input/datasets/tirthmanavadariya/vicmae-checkpoint/checkpoint_epoch0009_s2.pth"
dataset_ckpt_path = "/kaggle/input/vicmae-checkpoint/checkpoint_epoch0009_s2.pth"

candidate_paths = [
    user_ckpt_path,
    dataset_ckpt_path,
]

resume_ckpt = None
for p in candidate_paths:
    if os.path.exists(p):
        resume_ckpt = p
        break

if resume_ckpt is None:
    # Recursive search in /kaggle/input for checkpoint_epoch0009_s2.pth or any epoch 9 checkpoint
    found = glob.glob("/kaggle/input/**/checkpoint_epoch0009_s2.pth", recursive=True)
    if not found:
        found = glob.glob("/kaggle/input/**/checkpoint_epoch0009*.pth", recursive=True)
    if not found:
        found = glob.glob("/kaggle/input/**/checkpoint*.pth", recursive=True)
    if not found:
        found = glob.glob(f"{OUTPUT_DIR}/checkpoint_epoch0009*.pth")
    if found:
        resume_ckpt = found[0]

resume_flag = ""
if resume_ckpt and os.path.exists(resume_ckpt):
    ckpt_size_mb = os.path.getsize(resume_ckpt) / (1024**2)
    print(f"✅ Found resume checkpoint ({ckpt_size_mb:.1f} MB): {resume_ckpt}")
    print(f"🔁 Resuming Stage 2 training from Epoch {START_EPOCH} up to Epoch {TOTAL_EPOCHS}...")
    resume_flag = f"--resume '{resume_ckpt}' --start_epoch {START_EPOCH}"

    # Restore previous training logs if present alongside checkpoint
    ckpt_dir = os.path.dirname(resume_ckpt)
    for log_name in ["train_extended_log.json", "log_stage2.json"]:
        src_log = os.path.join(ckpt_dir, log_name)
        dst_log = os.path.join(OUTPUT_DIR, log_name)
        if os.path.exists(src_log) and not os.path.exists(dst_log):
            shutil.copy(src_log, dst_log)
            print(f"📋 Restored previous training log: {dst_log}")
else:
    print(f"⚠️ Checkpoint not found at: {user_ckpt_path}")
    print(f"⚠️ Also checked: {dataset_ckpt_path}")
    print("Files currently present in /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print("  ", os.path.join(root, f))
    raise FileNotFoundError(
        "Checkpoint 'checkpoint_epoch0009_s2.pth' not found in /kaggle/input! "
        "Please ensure your 'vicmae-checkpoint' dataset is attached in Kaggle's right sidebar."
    )

# -----------------------------------------------------------------
# 2. GPU Hardware & Batch Size Configuration
# -----------------------------------------------------------------
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 30:
        BATCH_SIZE = 32
        ACCUM_ITER = 2
    elif vram_gb >= 12:
        BATCH_SIZE = 16
        ACCUM_ITER = 4
    else:
        BATCH_SIZE = 8
        ACCUM_ITER = 8
else:
    BATCH_SIZE = 4
    ACCUM_ITER = 1

# Auto-detect user uploaded videos in /kaggle/input or /kaggle/working
vid_files = glob.glob("/kaggle/input/**/*.mp4", recursive=True) + glob.glob("/kaggle/working/**/*.mp4", recursive=True)
video_flag = ""
if vid_files:
    vroot = os.path.dirname(vid_files[0])
    video_flag = f"--video_root '{vroot}'"
    print(f"🎬 Auto-detected video inputs ({len(vid_files)} files): {video_flag}")
else:
    print("ℹ️ No video files detected. Training with static image preservation and delta_t=0 cross-reconstruction.")

print(f"Running with batch_size={BATCH_SIZE}, accum_iter={ACCUM_ITER}")

cmd = f"""cd {REPO_DIR} && python train_extended.py \
    --checkpoint {CKPT_PATH} \
    --image_root {IMAGE_DIR} \
    {video_flag} \
    {resume_flag} \
    --vicmae_arch vicmae_vit_base_patch16 \
    --batch_size {BATCH_SIZE} \
    --accum_iter {ACCUM_ITER} \
    --stage1_epochs 0 \
    --stage2_epochs {TOTAL_EPOCHS} \
    --save_freq 5 \
    --decoder_lr 1.5e-4 \
    --backbone_lr 1e-5 \
    --use_lora \
    --lora_r 16 \
    --use_amp \
    --lambda_cross 1.0 \
    --lambda_pres 1.0 \
    --output_dir {OUTPUT_DIR} \
    --num_workers 4
"""

!{cmd}

## 6. Visualize Training Losses & Feature Preservation
Plot the complete training curves across epochs 1 to 30.

In [ ]:
import json, os
import matplotlib.pyplot as plt
from pathlib import Path

log_file = Path("/kaggle/working/output/vicmae_extended/train_extended_log.json")
if log_file.exists():
    history_by_epoch = {}
    with open(log_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
                ep = entry.get("epoch")
                if ep is not None:
                    history_by_epoch[ep] = entry
            except Exception:
                pass

    if history_by_epoch:
        sorted_epochs = sorted(history_by_epoch.keys())
        epochs = sorted_epochs
        total_losses = [history_by_epoch[ep].get("loss_total", 0) for ep in sorted_epochs]
        mae_losses   = [history_by_epoch[ep].get("loss_mae", 0) for ep in sorted_epochs]
        pres_losses  = [history_by_epoch[ep].get("loss_pres", 0) for ep in sorted_epochs]
        lrs          = [history_by_epoch[ep].get("lr", 0) for ep in sorted_epochs]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(epochs, total_losses, label="Total Loss", color="tab:blue", lw=2, marker="o", markersize=3)
        ax1.plot(epochs, mae_losses, label="MAE Loss", color="tab:green", linestyle="--")
        ax1.plot(epochs, pres_losses, label="Preservation Loss", color="tab:orange", linestyle="--")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.set_title("Training Loss Decomposition (Complete History)")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, lrs, label="Learning Rate", color="tab:purple", lw=2)
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("LR")
        ax2.set_title("Learning Rate Schedule")
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("/kaggle/working/output/vicmae_extended/curves.png", dpi=150)
        plt.show()
        print("✅ Saved training curves to /kaggle/working/output/vicmae_extended/curves.png")
    else:
        print(f"No valid entries found in {log_file}.")
else:
    print(f"Log file {log_file} not found yet.")